In [16]:
ACTIVE_TASKS = {
    "contrastChangeDetection",
    "symbolSearch",
}

PASSIVE_TASKS = {
    "RestingState",
    "surroundSupp",
    "DespicableMe",
    "DiaryOfAWimpyKid",
    "FunwithFractals",
    "ThePresent",
}

In [17]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
bucket = "openneuro.org"

def list_subjects():
    resp = s3.list_objects_v2(Bucket=bucket, Prefix="ds005516/")
    subs = set()

    for obj in resp.get("Contents", []):
        key = obj["Key"]
        if key.startswith("ds005516/sub-"):
            parts = key.split("/")
            subs.add(parts[1])  # sub-XXXXX

    return sorted(subs)

In [18]:
import numpy as np

def window_raw(raw, label, window_sec=2.0, step_sec=2.0, max_windows=None):
    raw = raw.copy()
    raw.load_data()

    sfreq = raw.info["sfreq"]
    win = int(window_sec * sfreq)
    step = int(step_sec * sfreq)

    data = raw.get_data()  # (channels, time)

    X, y = [], []

    for start in range(0, data.shape[1] - win, step):
        segment = data[:, start:start + win]

        # basic normalization (important for stability)
        segment = (segment - segment.mean(axis=1, keepdims=True)) / (
            segment.std(axis=1, keepdims=True) + 1e-8
        )

        X.append(segment)
        y.append(label)

        if max_windows and len(X) >= max_windows:
            break

    return np.array(X), np.array(y)

In [19]:
TOTAL_WINDOWS = 6000  # example, adjust as needed

from collections import defaultdict

task_quota = {}

# active → 50%
active_total = TOTAL_WINDOWS // 2
per_active = active_total // len(ACTIVE_TASKS)

for t in ACTIVE_TASKS:
    task_quota[t] = per_active

# passive → 50%
passive_total = TOTAL_WINDOWS // 2
per_passive = passive_total // len(PASSIVE_TASKS)

for t in PASSIVE_TASKS:
    task_quota[t] = per_passive

In [20]:
def list_eeg_files(subject):
    prefix = f"ds005516/{subject}/eeg/"
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)

    files = []
    for obj in resp.get("Contents", []):
        key = obj["Key"]

        if key.endswith("_eeg.set"):
            files.append(key)

    return files

def parse_task(key):
    # example: sub-XXX_task-contrastChangeDetection_run-1_eeg.set
    fname = key.split("/")[-1]

    for part in fname.split("_"):
        if part.startswith("task-"):
            return part.replace("task-", "")
    return None

def get_label(task):
    if task in ACTIVE_TASKS:
        return 1
    elif task in PASSIVE_TASKS:
        return 0
    else:
        return None  # ignore unknown

import tempfile
import os
import mne

def load_recording(set_key):
    base = set_key[:-4]  # remove .set

    with tempfile.TemporaryDirectory() as tmpdir:
        set_path = os.path.join(tmpdir, "data.set")
        fdt_path = os.path.join(tmpdir, "data.fdt")

        s3.download_file(bucket, base + ".set", set_path)
        s3.download_file(bucket, base + ".fdt", fdt_path)

        raw = mne.io.read_raw_eeglab(set_path, preload=False)
        return raw

def data_generator(subjects):
    for sub in subjects:
        eeg_files = list_eeg_files(sub)

        for key in eeg_files:
            task = parse_task(key)
            label = get_label(task)

            if label is None:
                continue

            try:
                raw = load_recording(key)

                yield {
                    "subject": sub,
                    "task": task,
                    "label": label,
                    "raw": raw
                }

            except Exception as e:
                print(f"Skipping {key}: {e}")

In [21]:
def build_dataset(subjects, total_windows=6000):
    X_all = []
    y_all = []

    collected = {t: 0 for t in task_quota}

    for sample in data_generator(subjects):
        task = sample["task"]
        raw = sample["raw"]

        if task not in task_quota:
            continue

        if collected[task] >= task_quota[task]:
            continue

        label = 1 if task in ACTIVE_TASKS else 0

        remaining = task_quota[task] - collected[task]

        X, y = window_raw(
            raw,
            label,
            window_sec=2.0,
            step_sec=2.0,
            max_windows=remaining
        )

        if len(X) == 0:
            continue

        X_all.append(X)
        y_all.append(y)

        collected[task] += len(X)

        # early stop if all quotas met
        if all(collected[t] >= task_quota[t] for t in task_quota):
            break

    X_all = np.concatenate(X_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)

    return X_all, y_all

In [22]:
subjects = list_subjects()[:30]

def shuffle_dataset(X, y):
    idx = np.random.permutation(len(X))
    return X[idx], y[idx]

X, y = build_dataset(subjects, total_windows=6000)
X, y = shuffle_dataset(X, y)

print(X.shape, y.shape)

Skipping ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-DespicableMe_eeg.set: An error occurred (404) when calling the HeadObject operation: Not Found
Skipping ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-DiaryOfAWimpyKid_eeg.set: An error occurred (404) when calling the HeadObject operation: Not Found
Skipping ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-FunwithFractals_eeg.set: An error occurred (404) when calling the HeadObject operation: Not Found
Skipping ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-RestingState_eeg.set: An error occurred (404) when calling the HeadObject operation: Not Found
Skipping ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-ThePresent_eeg.set: An error occurred (404) when calling the HeadObject operation: Not Found


KeyboardInterrupt: 

In [32]:
!cd ~

In [34]:
!ls -la

total 0
drwxr-xr-x@ 2 roman  staff  64 May  5 18:33 .
drwxr-xr-x@ 2 roman  staff  64 May  5 18:34 ..


In [30]:
!datalad install https://github.com/OpenNeuroDatasets/ds005516.git

Traceback (most recent call last):
  File "/Users/roman/PycharmProjects/brain_data/.venv/bin/datalad", line 5, in <module>
    from datalad.cli.main import main
  File "/Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/datalad/__init__.py", line 125, in <module>
    cfg = ConfigManager()
          ^^^^^^^^^^^^^^^
  File "/Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/datalad/config.py", line 399, in __init__
    self.reload(force=True)
  File "/Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/datalad/config.py", line 460, in reload
    self._stores[store_id] = self._reload(runargs)
                             ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/datalad/config.py", line 488, in _reload
    stdout, stderr = self._run(
                     ^^^^^^^^^^
  File "/Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/datalad/con

In [31]:
%cd ds005516

[Errno 2] No such file or directory: 'ds005516'
/Users/roman/PycharmProjects/brain_data/experiments_eeg


In [26]:
!datalad get sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-RestingState_eeg.set

Total:   0%|                                   | 0.00/90.7M [00:00<?, ? Bytes/s]
Get sub-NDAR .. tate_eeg.set:   0%|            | 0.00/90.7M [00:00<?, ? Bytes/s]
Get sub-NDAR .. tate_eeg.set:   0%|    | 33.3k/90.7M [00:00<04:51, 311k Bytes/s]
Get sub-NDAR .. tate_eeg.set:   0%|    | 68.1k/90.7M [00:00<04:57, 305k Bytes/s]
Get sub-NDAR .. tate_eeg.set:   0%|     | 120k/90.7M [00:00<03:50, 393k Bytes/s]
Get sub-NDAR .. tate_eeg.set:   0%|     | 207k/90.7M [00:00<02:45, 548k Bytes/s]
Get sub-NDAR .. tate_eeg.set:   0%|     | 347k/90.7M [00:00<01:49, 828k Bytes/s]
Get sub-NDAR .. tate_eeg.set:   1%|    | 573k/90.7M [00:00<01:10, 1.27M Bytes/s]
Get sub-NDAR .. tate_eeg.set:   1%|    | 817k/90.7M [00:00<00:56, 1.60M Bytes/s]
Get sub-NDAR .. tate_eeg.set:   1%|   | 1.03M/90.7M [00:00<00:51, 1.74M Bytes/s]
Get sub-NDAR .. tate_eeg.set:   1%|   | 1.29M/90.7M [00:00<00:45, 1.96M Bytes/s]
Get sub-NDAR .. tate_eeg.set:   2%|   | 1.53M/90.7M [00:01<00:42, 2.10M Bytes/s]
Get sub-NDAR .. tate_eeg.set

# actual begin

In [40]:
!cd ~

In [41]:
!ls -la

total 0
drwxr-xr-x@ 2 roman  staff  64 May  5 18:33 .
drwxr-xr-x@ 2 roman  staff  64 May  5 18:34 ..


In [37]:
os.listdir(".")

[]

In [42]:
import subprocess
import os

# already present subjects (no need for git listing now)
subjects = sorted([
    d for d in os.listdir("/Users/roman/PycharmProjects/brain_data/ds005516")
    if d.startswith("sub-")
])[:30]

subjects

['sub-NDARAB678VYW',
 'sub-NDARAB683CYD',
 'sub-NDARAC296UCB',
 'sub-NDARAD459XJK',
 'sub-NDARAG429CGW',
 'sub-NDARAG788YV9',
 'sub-NDARAJ182PTB',
 'sub-NDARAJ401AX3',
 'sub-NDARAJ674WJT',
 'sub-NDARAK772VFJ',
 'sub-NDARAM177DKJ',
 'sub-NDARAM946HJE',
 'sub-NDARAN302EEM',
 'sub-NDARAP283ZBW',
 'sub-NDARAP748AWX',
 'sub-NDARAU517MC6',
 'sub-NDARAW427GWK',
 'sub-NDARAW620GJ8',
 'sub-NDARAY761BFH',
 'sub-NDARAY977BZT',
 'sub-NDARAZ532KK0',
 'sub-NDARBB542URX',
 'sub-NDARBE096YK6',
 'sub-NDARBF402JLH',
 'sub-NDARBH275EXT',
 'sub-NDARBH482JK1',
 'sub-NDARBH789CUP',
 'sub-NDARBK194FF5',
 'sub-NDARBP713ZWA',
 'sub-NDARBT780KVC']

In [45]:
for sub in subjects:
    subprocess.run(["datalad", "get", "--recursive", sub], check=False)

Traceback (most recent call last):
  File "/Users/roman/PycharmProjects/brain_data/.venv/bin/datalad", line 5, in <module>
    from datalad.cli.main import main
  File "/Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/datalad/__init__.py", line 125, in <module>
    cfg = ConfigManager()
          ^^^^^^^^^^^^^^^
  File "/Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/datalad/config.py", line 399, in __init__
    self.reload(force=True)
  File "/Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/datalad/config.py", line 460, in reload
    self._stores[store_id] = self._reload(runargs)
                             ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/datalad/config.py", line 488, in _reload
    stdout, stderr = self._run(
                     ^^^^^^^^^^
  File "/Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/datalad/con

In [43]:


def list_eeg_files(subject):
    eeg_dir = os.path.join(subject, "eeg")
    if not os.path.exists(eeg_dir):
        return []

    return [
        os.path.join(eeg_dir, f)
        for f in os.listdir(eeg_dir)
        if f.endswith("_eeg.set")
    ]

def select_run1(files):
    groups = {}

    for f in files:
        name = os.path.basename(f)

        task = None
        for part in name.split("_"):
            if part.startswith("task-"):
                task = part.replace("task-", "")
                break

        if task is None:
            continue

        base = task.split("-")[0]
        groups.setdefault(base, []).append(f)

    selected = []
    for base, g in groups.items():
        run1 = [x for x in g if "run-1" in x]
        selected.append(run1[0] if run1 else sorted(g)[0])

    return selected


for sub in subjects:
    print("Downloading:", sub)

    eeg_files = list_eeg_files(sub)
    eeg_files = select_run1(eeg_files)

    for f in eeg_files:
        try:
            subprocess.run(["datalad", "get", f], check=True)
        except subprocess.CalledProcessError:
            print("Skip:", f)

Downloading: sub-NDARAB678VYW
Downloading: sub-NDARAB683CYD
Downloading: sub-NDARAC296UCB
Downloading: sub-NDARAD459XJK
Downloading: sub-NDARAG429CGW
Downloading: sub-NDARAG788YV9
Downloading: sub-NDARAJ182PTB
Downloading: sub-NDARAJ401AX3
Downloading: sub-NDARAJ674WJT
Downloading: sub-NDARAK772VFJ
Downloading: sub-NDARAM177DKJ
Downloading: sub-NDARAM946HJE
Downloading: sub-NDARAN302EEM
Downloading: sub-NDARAP283ZBW
Downloading: sub-NDARAP748AWX
Downloading: sub-NDARAU517MC6
Downloading: sub-NDARAW427GWK
Downloading: sub-NDARAW620GJ8
Downloading: sub-NDARAY761BFH
Downloading: sub-NDARAY977BZT
Downloading: sub-NDARAZ532KK0
Downloading: sub-NDARBB542URX
Downloading: sub-NDARBE096YK6
Downloading: sub-NDARBF402JLH
Downloading: sub-NDARBH275EXT
Downloading: sub-NDARBH482JK1
Downloading: sub-NDARBH789CUP
Downloading: sub-NDARBK194FF5
Downloading: sub-NDARBP713ZWA
Downloading: sub-NDARBT780KVC


In [ ]:
sub-NDARAB678VYW
eeg
sub-NDARAB678VYW_task-contrastChangeDetection_run-1_channels.tsv
sub-NDARAB678VYW_task-contrastChangeDetection_run-1_eeg.json
sub-NDARAB678VYW_task-contrastChangeDetection_run-1_eeg.set
sub-NDARAB678VYW_task-contrastChangeDetection_run-1_events.tsv
sub-NDARAB678VYW_task-contrastChangeDetection_run-2_channels.tsv
sub-NDARAB678VYW_task-contrastChangeDetection_run-2_eeg.json
sub-NDARAB678VYW_task-contrastChangeDetection_run-2_eeg.set
sub-NDARAB678VYW_task-contrastChangeDetection_run-2_events.tsv
sub-NDARAB678VYW_task-contrastChangeDetection_run-3_channels.tsv
sub-NDARAB678VYW_task-contrastChangeDetection_run-3_eeg.json
sub-NDARAB678VYW_task-contrastChangeDetection_run-3_eeg.set

In [1]:
ACTIVE_TASKS = {
    "contrastChangeDetection",
    "symbolSearch",
}

PASSIVE_TASKS = {
    "RestingState",
    "surroundSupp",
    "DespicableMe",
    "DiaryOfAWimpyKid",
    "FunwithFractals",
    "ThePresent",
}

import numpy as np

def window_raw(raw, label, window_sec=2.0, step_sec=2.0, max_windows=None):
    raw = raw.copy().load_data()

    sfreq = raw.info["sfreq"]
    win = int(window_sec * sfreq)
    step = int(step_sec * sfreq)

    data = raw.get_data()

    X, y = [], []

    for start in range(0, data.shape[1] - win, step):
        segment = data[:, start:start + win]

        segment = (segment - segment.mean(axis=1, keepdims=True)) / (
            segment.std(axis=1, keepdims=True) + 1e-8
        )

        X.append(segment)
        y.append(label)

        if max_windows and len(X) >= max_windows:
            break

    return np.array(X), np.array(y)

import os

def list_subjects(root="."):
    return sorted([
        d for d in os.listdir(root)
        if d.startswith("sub-") and os.path.isdir(os.path.join(root, d))
    ])

def list_eeg_files(subject):
    eeg_dir = os.path.join(subject, "eeg")
    if not os.path.exists(eeg_dir):
        return []

    return [
        os.path.join(eeg_dir, f)
        for f in os.listdir(eeg_dir)
        if f.endswith("_eeg.set")
    ]

def parse_task(path):
    fname = os.path.basename(path)

    for part in fname.split("_"):
        if part.startswith("task-"):
            return part.replace("task-", "")
    return None

def get_label(task):
    if task in ACTIVE_TASKS:
        return 1
    elif task in PASSIVE_TASKS:
        return 0
    return None

import subprocess

def datalad_get(path):
    subprocess.run(["datalad", "get", path], check=True)

    import mne

def load_recording(set_path):
    # fetch .set (+ automatically .fdt)
    datalad_get(set_path)

    raw = mne.io.read_raw_eeglab(set_path, preload=False)
    return raw

def build_dataset(subjects):
    X_all, y_all = [], []
    collected = {t: 0 for t in task_quota}

    for sub in subjects:
        eeg_files = list_eeg_files(sub)

        for path in eeg_files:
            task = parse_task(path)
            label = get_label(task)

            if task not in task_quota or label is None:
                continue

            if collected[task] >= task_quota[task]:
                continue

            try:
                raw = load_recording(path)

                remaining = task_quota[task] - collected[task]

                X, y = window_raw(
                    raw,
                    label,
                    window_sec=2.0,
                    step_sec=2.0,
                    max_windows=remaining
                )

                if len(X) == 0:
                    continue

                X_all.append(X)
                y_all.append(y)

                collected[task] += len(X)

                # optional: free disk (important in Colab)
                subprocess.run(["datalad", "drop", path], stdout=subprocess.DEVNULL)

                if all(collected[t] >= task_quota[t] for t in task_quota):
                    return np.concatenate(X_all), np.concatenate(y_all)

            except Exception as e:
                print(f"Skipping {path}: {e}")

    return np.concatenate(X_all), np.concatenate(y_all)

# dataset building

In [29]:
TOTAL_WINDOWS = 6000

task_quota = {}

active_total = TOTAL_WINDOWS // 2
per_active = active_total // len(ACTIVE_TASKS)

for t in ACTIVE_TASKS:
    task_quota[t] = per_active

passive_total = TOTAL_WINDOWS // 2
per_passive = passive_total // len(PASSIVE_TASKS)

for t in PASSIVE_TASKS:
    task_quota[t] = per_passive

In [30]:
subjects = list_subjects()[:30]

X, y = build_dataset(subjects)

# shuffle
idx = np.random.permutation(len(X))
X, y = X[idx], y[idx]

print(X.shape, y.shape)

get(ok): sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-DespicableMe_eeg.set (file) [from s3-PUBLIC...]
Reading /Users/roman/PycharmProjects/brain_data/experiments_eeg/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-DespicableMe_eeg.set
Reading 0 ... 86139  =      0.000 ...   172.278 secs...


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_80166/156923057.py:88: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=False)


get(ok): sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-ThePresent_eeg.set (file) [from s3-PUBLIC...]
Reading /Users/roman/PycharmProjects/brain_data/experiments_eeg/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-ThePresent_eeg.set
Reading 0 ... 102439  =      0.000 ...   204.878 secs...
get(ok): sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-FunwithFractals_eeg.set (file) [from s3-PUBLIC...]
Reading /Users/roman/PycharmProjects/brain_data/experiments_eeg/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-FunwithFractals_eeg.set
Reading 0 ... 82352  =      0.000 ...   164.704 secs...


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_80166/156923057.py:88: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=False)


get(ok): sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-contrastChangeDetection_run-1_eeg.set (file) [from s3-PUBLIC...]
Reading /Users/roman/PycharmProjects/brain_data/experiments_eeg/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-contrastChangeDetection_run-1_eeg.set
Reading 0 ... 179228  =      0.000 ...   358.456 secs...
get(ok): sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-surroundSupp_run-1_eeg.set (file) [from s3-PUBLIC...]
Reading /Users/roman/PycharmProjects/brain_data/experiments_eeg/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-surroundSupp_run-1_eeg.set
Reading 0 ... 139316  =      0.000 ...   278.632 secs...


...

get(ok): sub-NDARBF402JLH/eeg/sub-NDARBF402JLH_task-symbolSearch_eeg.set (file) [from s3-PUBLIC...]
Reading /Users/roman/PycharmProjects/brain_data/experiments_eeg/ds005516/sub-NDARBF402JLH/eeg/sub-NDARBF402JLH_task-symbolSearch_eeg.set
Reading 0 ... 87413  =      0.000 ...   174.826 secs...


/Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/pymatreader/utils.py:342: UserWarning: pymatreader cannot import Matlab string variables. Please convert these variables to char arrays in Matlab.
  warn(
/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_80166/156923057.py:88: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=False)


get(ok): sub-NDARBH275EXT/eeg/sub-NDARBH275EXT_task-symbolSearch_eeg.set (file) [from s3-PUBLIC...]
Reading /Users/roman/PycharmProjects/brain_data/experiments_eeg/ds005516/sub-NDARBH275EXT/eeg/sub-NDARBH275EXT_task-symbolSearch_eeg.set
Reading 0 ... 91241  =      0.000 ...   182.482 secs...


/Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/pymatreader/utils.py:342: UserWarning: pymatreader cannot import Matlab string variables. Please convert these variables to char arrays in Matlab.
  warn(
/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_80166/156923057.py:88: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=False)


get(ok): sub-NDARBK194FF5/eeg/sub-NDARBK194FF5_task-symbolSearch_eeg.set (file) [from s3-PUBLIC...]
Reading /Users/roman/PycharmProjects/brain_data/experiments_eeg/ds005516/sub-NDARBK194FF5/eeg/sub-NDARBK194FF5_task-symbolSearch_eeg.set
Reading 0 ... 139136  =      0.000 ...   278.272 secs...
(6000, 129, 1000) (6000,)


In [31]:
X.shape, y.shape

((6000, 129, 1000), (6000,))

# pipeline

In [35]:
import numpy as np
from scipy.linalg import fractional_matrix_power
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from pyriemann.utils.mean import mean_covariance
from pyriemann.tangentspace import TangentSpace

# ============================================================
# CONFIG
# ============================================================

REG = 1e-5
TARGET_SFREQ = 125
WINDOW_SEC = 2.0
WINDOW_LEN = int(WINDOW_SEC * TARGET_SFREQ)  # Should match your window size

# ============================================================
# COVARIANCE ESTIMATION
# ============================================================

def compute_covariance(w):
    """
    Compute regularized covariance matrix from window
    w: shape (n_channels, n_samples)
    """
    w = w - np.mean(w, axis=1, keepdims=True)
    s = w.shape[1]
    C = (w @ w.T) / (s - 1)
    C += REG * np.eye(C.shape[0])
    return C

# ============================================================
# HJORTH FEATURES
# ============================================================

def hjorth_activity(x):
    return np.var(x)

def hjorth_mobility(x):
    v = np.var(x)
    return np.sqrt(np.var(np.diff(x)) / (v + 1e-12))

def hjorth_complexity(x):
    m = hjorth_mobility(x)
    return hjorth_mobility(np.diff(x)) / (m + 1e-12)

def extract_hjorth_features_from_window(w):
    """
    Extract Hjorth features from a single window across all channels
    w: shape (n_channels, n_samples)
    Returns: 1D array of features (n_channels * 5)
    """
    n_channels = w.shape[0]
    features = []

    for ch in range(n_channels):
        x = w[ch]

        a = hjorth_activity(x)
        m = hjorth_mobility(x)
        c = hjorth_complexity(x)

        features.extend([
            a, m, c,
            np.log(a + 1e-12),  # log activity
            np.log(m + 1e-12)   # log mobility
        ])

    return np.array(features)

# ============================================================
# PROCESS ALL WINDOWS
# ============================================================

print("="*60)
print("PROCESSING WINDOWS")
print("="*60)

# X: (6000, 129, 1000) - 129 channels, 1000 samples (8 seconds at 125Hz? Let me check)
# Actually 1000 samples / 125 Hz = 8 seconds, but your window_sec=2.0 means 250 samples
# Let me check the shape

print(f"Original X shape: {X.shape}")
print(f"  Number of windows: {X.shape[0]}")
print(f"  Number of channels: {X.shape[1]}")
print(f"  Samples per window: {X.shape[2]}")

# If your windows are 2 seconds, samples should be 250 (2.0 * 125)
# If they are 8 seconds (1000 samples), you may want to sub-window
# Let's assume they are 2-second windows (250 samples) or adjust

n_channels = X.shape[1]
n_samples_per_window = X.shape[2]
expected_samples = WINDOW_LEN  # 250

if n_samples_per_window != expected_samples:
    print(f"\n⚠️  Warning: Window has {n_samples_per_window} samples, expected {expected_samples}")
    print(f"   Adjusting WINDOW_LEN to match data: {n_samples_per_window}")
    WINDOW_LEN = n_samples_per_window

print(f"\nExtracting features from {X.shape[0]} windows...")

all_covs = []
all_hjorth = []

for i, w in enumerate(X):
    if (i + 1) % 1000 == 0:
        print(f"  Processed {i+1}/{X.shape[0]} windows")

    # Extract Hjorth features
    hjorth_feats = extract_hjorth_features_from_window(w)
    all_hjorth.append(hjorth_feats)

    # Compute covariance matrix
    C = compute_covariance(w)
    all_covs.append(C)

all_hjorth = np.array(all_hjorth)
all_covs = np.array(all_covs)

print(f"\nHjorth features shape: {all_hjorth.shape}")
print(f"Covariance matrices shape: {all_covs.shape}")

# ============================================================
# SUBJECT SPLIT
# ============================================================
# Since you have no subject labels in this processed dataset,
# we'll use stratified k-fold cross-validation instead

print("\n" + "="*60)
print("CROSS-VALIDATION SETUP")
print("="*60)

n_samples = len(y)
n_folds = 5

skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

baseline_aucs = []
final_aucs = []

print(f"Running {n_folds}-fold cross-validation...")

fold = 1
for train_idx, test_idx in skf.split(np.zeros(n_samples), y):
    print(f"\n{'='*50}")
    print(f"FOLD {fold}/{n_folds}")
    print(f"{'='*50}")

    covs_train = all_covs[train_idx]
    covs_test = all_covs[test_idx]
    hjorth_train = all_hjorth[train_idx]
    hjorth_test = all_hjorth[test_idx]
    y_train = y[train_idx]
    y_test = y[test_idx]

    # ---- Riemannian mean ----
    print("  Computing Riemannian mean...")
    M = mean_covariance(covs_train, metric='riemann')

    # ---- Whitening ----
    M_inv_sqrt = fractional_matrix_power(M, -0.5)
    covs_train_w = np.array([M_inv_sqrt @ C @ M_inv_sqrt for C in covs_train])
    covs_test_w = np.array([M_inv_sqrt @ C @ M_inv_sqrt for C in covs_test])

    # ---- Tangent space projection ----
    ts = TangentSpace(metric='riemann')
    ts.fit(covs_train_w)

    X_ts_train = ts.transform(covs_train_w)
    X_ts_test = ts.transform(covs_test_w)

    # ---- Standardize Hjorth features ----
    scaler = StandardScaler()
    X_hjorth_train = scaler.fit_transform(hjorth_train)
    X_hjorth_test = scaler.transform(hjorth_test)

    # ---- Fuse features ----
    X_train = np.concatenate([X_hjorth_train, X_ts_train], axis=1)
    X_test = np.concatenate([X_hjorth_test, X_ts_test], axis=1)

    print(f"  Features: {X_train.shape[1]} (Hjorth: {X_hjorth_train.shape[1]}, Riemannian: {X_ts_train.shape[1]})")

    # ---- Train XGBoost ----
    model = XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        n_jobs=-1,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_prob)
    final_aucs.append(auc)

    print(f"\n  Fold {fold} Results:")
    print(f"    Accuracy: {np.mean(y_pred == y_test):.4f}")
    print(f"    ROC-AUC: {auc:.4f}")
    print(f"    Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))

    fold += 1

# ============================================================
# FINAL RESULTS
# ============================================================
print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(f"ROC-AUC across {n_folds} folds: {np.mean(final_aucs):.4f} (+/- {np.std(final_aucs):.4f})")
print(f"Individual fold AUCs: {[f'{x:.4f}' for x in final_aucs]}")

PROCESSING WINDOWS
Original X shape: (6000, 129, 1000)
  Number of windows: 6000
  Number of channels: 129
  Samples per window: 1000

⚠️  Warning: Window has 1000 samples, expected 250
   Adjusting WINDOW_LEN to match data: 1000

Extracting features from 6000 windows...
  Processed 1000/6000 windows
  Processed 2000/6000 windows
  Processed 3000/6000 windows
  Processed 4000/6000 windows
  Processed 5000/6000 windows
  Processed 6000/6000 windows

Hjorth features shape: (6000, 645)
Covariance matrices shape: (6000, 129, 129)

CROSS-VALIDATION SETUP
Running 5-fold cross-validation...

FOLD 1/5
  Computing Riemannian mean...


/Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/pyriemann/utils/base.py:51: UserWarning: Convergence not reached
  return func(X, *args, **kwargs)


  Features: 9030 (Hjorth: 645, Riemannian: 8385)

  Fold 1 Results:
    Accuracy: 0.9558
    ROC-AUC: 0.9901
    Classification Report:
              precision    recall  f1-score   support

           0     0.9740    0.9367    0.9550       600
           1     0.9390    0.9750    0.9567       600

    accuracy                         0.9558      1200
   macro avg     0.9565    0.9558    0.9558      1200
weighted avg     0.9565    0.9558    0.9558      1200


FOLD 2/5
  Computing Riemannian mean...


KeyboardInterrupt: 

# split

In [39]:
import numpy as np

def split_subjects(subjects, test_ratio=0.3, seed=42):
    subjects = np.array(subjects)
    rng = np.random.RandomState(seed)
    rng.shuffle(subjects)

    n_test = int(len(subjects) * test_ratio)

    return subjects[n_test:].tolist(), subjects[:n_test].tolist()

MAX_PER_TASK = 6000

def compute_task_quota():
    task_quota = {}

    for t in ACTIVE_TASKS:
        task_quota[t] = MAX_PER_TASK

    for t in PASSIVE_TASKS:
        task_quota[t] = MAX_PER_TASK

    return task_quota

import subprocess

def build_dataset_streaming(subjects, task_quota):
    X_all, y_all = [], []
    collected = {t: 0 for t in task_quota}

    for sub in subjects:
        eeg_files = list_eeg_files(sub)

        for path in eeg_files:
            task = parse_task(path)
            label = get_label(task)

            if task not in task_quota or label is None:
                continue

            if collected[task] >= task_quota[task]:
                continue

            try:
                raw = load_recording(path)

                remaining = task_quota[task] - collected[task]

                X, y = window_raw(
                    raw,
                    label,
                    window_sec=2.0,
                    step_sec=2.0,
                    max_windows=remaining
                )

                if len(X) == 0:
                    continue

                X_all.append(X)
                y_all.append(y)

                collected[task] += len(X)

                subprocess.run(
                    ["datalad", "drop", path],
                    stdout=subprocess.DEVNULL
                )

                # early stop
                if all(collected[t] >= task_quota[t] for t in task_quota):
                    return np.concatenate(X_all), np.concatenate(y_all)

            except Exception as e:
                print(f"Skipping {path}: {e}")

    return np.concatenate(X_all), np.concatenate(y_all)

## build

In [ ]:
subjects = list_subjects()[:30]

train_subjects, test_subjects = split_subjects(subjects, test_ratio=0.3)

task_quota = compute_task_quota()

X_train, y_train = build_dataset_streaming(train_subjects, task_quota)
X_test, y_test   = build_dataset_streaming(test_subjects, task_quota)

# shuffle
def shuffle(X, y):
    idx = np.random.permutation(len(X))
    return X[idx], y[idx]

X_train, y_train = shuffle(X_train, y_train)
X_test, y_test = shuffle(X_test, y_test)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

get(ok): sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-DespicableMe_eeg.set (file) [from s3-PUBLIC...]
Reading /Users/roman/PycharmProjects/brain_data/experiments_eeg/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-DespicableMe_eeg.set
Reading 0 ... 86139  =      0.000 ...   172.278 secs...


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_80166/156923057.py:88: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=False)


get(ok): sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-ThePresent_eeg.set (file) [from s3-PUBLIC...]
Reading /Users/roman/PycharmProjects/brain_data/experiments_eeg/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-ThePresent_eeg.set
Reading 0 ... 102439  =      0.000 ...   204.878 secs...
get(ok): sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-FunwithFractals_eeg.set (file) [from s3-PUBLIC...]
Reading /Users/roman/PycharmProjects/brain_data/experiments_eeg/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-FunwithFractals_eeg.set
Reading 0 ... 82352  =      0.000 ...   164.704 secs...


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_80166/156923057.py:88: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=False)


get(ok): sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-contrastChangeDetection_run-1_eeg.set (file) [from s3-PUBLIC...]
Reading /Users/roman/PycharmProjects/brain_data/experiments_eeg/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-contrastChangeDetection_run-1_eeg.set
Reading 0 ... 179228  =      0.000 ...   358.456 secs...
get(ok): sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-surroundSupp_run-1_eeg.set (file) [from s3-PUBLIC...]
Reading /Users/roman/PycharmProjects/brain_data/experiments_eeg/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-surroundSupp_run-1_eeg.set
Reading 0 ... 139316  =      0.000 ...   278.632 secs...


...

get(ok): sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-surroundSupp_run-2_eeg.set (file) [from s3-PUBLIC...]
Reading /Users/roman/PycharmProjects/brain_data/experiments_eeg/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-surroundSupp_run-2_eeg.set
Reading 0 ... 142176  =      0.000 ...   284.352 secs...
get(ok): sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-contrastChangeDetection_run-2_eeg.set (file) [from s3-PUBLIC...]
Reading /Users/roman/PycharmProjects/brain_data/experiments_eeg/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-contrastChangeDetection_run-2_eeg.set
Reading 0 ... 119483  =      0.000 ...   238.966 secs...
get(ok): sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-DiaryOfAWimpyKid_eeg.set (file) [from s3-PUBLIC...]
Reading /Users/roman/PycharmProjects/brain_data/experiments_eeg/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-DiaryOfAWimpyKid_eeg.set
Reading 0 ... 59532  =      0.000 ...   119.064 secs...


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_80166/156923057.py:88: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=False)


get(ok): sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-RestingState_eeg.set (file) [from s3-PUBLIC...]
Reading /Users/roman/PycharmProjects/brain_data/experiments_eeg/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-RestingState_eeg.set
Reading 0 ... 172890  =      0.000 ...   345.780 secs...


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_80166/156923057.py:88: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=False)


get(ok): sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-contrastChangeDetection_run-3_eeg.set (file) [from s3-PUBLIC...]
Reading /Users/roman/PycharmProjects/brain_data/experiments_eeg/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-contrastChangeDetection_run-3_eeg.set
Reading 0 ... 124250  =      0.000 ...   248.500 secs...


## pipeline

In [ ]:
import numpy as np
import mne
from scipy.linalg import fractional_matrix_power
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier
from pyriemann.utils.mean import mean_covariance
from pyriemann.tangentspace import TangentSpace

# ============================================================
# CONFIG
# ============================================================

REG = 1e-5
TARGET_SFREQ = 125
WINDOW_SEC = 2.0
WINDOW_LEN = int(WINDOW_SEC * TARGET_SFREQ)  # 250 samples

# ============================================================
# COVARIANCE ESTIMATION
# ============================================================

def compute_covariance(w):
    """
    Compute regularized covariance matrix from window
    w: shape (n_channels, n_samples)
    """
    w = w - np.mean(w, axis=1, keepdims=True)
    s = w.shape[1]
    C = (w @ w.T) / (s - 1)
    C += REG * np.eye(C.shape[0])
    return C

# ============================================================
# HJORTH FEATURES
# ============================================================

def hjorth_activity(x):
    return np.var(x)

def hjorth_mobility(x):
    v = np.var(x)
    return np.sqrt(np.var(np.diff(x)) / (v + 1e-12))

def hjorth_complexity(x):
    m = hjorth_mobility(x)
    return hjorth_mobility(np.diff(x)) / (m + 1e-12)

def extract_hjorth_features_from_window(w):
    """
    Extract Hjorth features from a single window across all channels
    w: shape (n_channels, n_samples)
    Returns: 1D array of features (n_channels * 5)
    """
    n_channels = w.shape[0]
    features = []

    for ch in range(n_channels):
        x = w[ch]

        a = hjorth_activity(x)
        m = hjorth_mobility(x)
        c = hjorth_complexity(x)

        features.extend([
            a, m, c,
            np.log(a + 1e-12),  # log activity
            np.log(m + 1e-12)   # log mobility
        ])

    return np.array(features)

# ============================================================
# PROCESS TRAINING DATA
# ============================================================

print("="*60)
print("PROCESSING TRAINING DATA")
print("="*60)
print(f"Training windows: {X_train.shape[0]}")
print(f"  Channels: {X_train.shape[1]}, Samples: {X_train.shape[2]}")

# Check if window length matches expected
if X_train.shape[2] != WINDOW_LEN:
    print(f"⚠️  Warning: Window has {X_train.shape[2]} samples, expected {WINDOW_LEN}")
    print(f"   Adjusting WINDOW_LEN to match data: {X_train.shape[2]}")
    WINDOW_LEN = X_train.shape[2]

n_channels = X_train.shape[1]

print(f"\nExtracting features from training windows...")

train_covs = []
train_hjorth = []

for i, w in enumerate(X_train):
    if (i + 1) % 1000 == 0:
        print(f"  Processed {i+1}/{X_train.shape[0]} training windows")

    # Extract Hjorth features
    hjorth_feats = extract_hjorth_features_from_window(w)
    train_hjorth.append(hjorth_feats)

    # Compute covariance matrix
    C = compute_covariance(w)
    train_covs.append(C)

train_hjorth = np.array(train_hjorth)
train_covs = np.array(train_covs)

print(f"\nTraining Hjorth features shape: {train_hjorth.shape}")
print(f"Training covariance matrices shape: {train_covs.shape}")

# ============================================================
# PROCESS TESTING DATA
# ============================================================

print("\n" + "="*60)
print("PROCESSING TESTING DATA")
print("="*60)
print(f"Testing windows: {X_test.shape[0]}")

test_covs = []
test_hjorth = []

for i, w in enumerate(X_test):
    if (i + 1) % 1000 == 0:
        print(f"  Processed {i+1}/{X_test.shape[0]} testing windows")

    # Extract Hjorth features
    hjorth_feats = extract_hjorth_features_from_window(w)
    test_hjorth.append(hjorth_feats)

    # Compute covariance matrix
    C = compute_covariance(w)
    test_covs.append(C)

test_hjorth = np.array(test_hjorth)
test_covs = np.array(test_covs)

print(f"\nTesting Hjorth features shape: {test_hjorth.shape}")
print(f"Testing covariance matrices shape: {test_covs.shape}")

# ============================================================
# WHITENING FUNCTION
# ============================================================

def whiten(C, M):
    M_inv_sqrt = fractional_matrix_power(M, -0.5)
    return M_inv_sqrt @ C @ M_inv_sqrt

# ============================================================
# RIEMANNIAN PIPELINE
# ============================================================

print("\n" + "="*60)
print("RIEMANNIAN PIPELINE")
print("="*60)

# ---- Riemannian mean on training covariances ----
print("\n[Step 1] Computing Riemannian mean on training data...")
M = mean_covariance(train_covs, metric='riemann')

# ---- Whitening ----
print("[Step 2] Whitening covariance matrices...")
train_covs_w = np.array([whiten(C, M) for C in train_covs])
test_covs_w = np.array([whiten(C, M) for C in test_covs])

# ---- Tangent space projection ----
print("[Step 3] Projecting to tangent space...")
ts = TangentSpace(metric='riemann')
ts.fit(train_covs_w)

X_ts_train = ts.transform(train_covs_w)
X_ts_test = ts.transform(test_covs_w)

print(f"  Tangent space features: {X_ts_train.shape[1]}")

# ---- Standardize Hjorth features ----
print("[Step 4] Standardizing Hjorth features...")
scaler = StandardScaler()
X_hjorth_train = scaler.fit_transform(train_hjorth)
X_hjorth_test = scaler.transform(test_hjorth)

print(f"  Hjorth features: {X_hjorth_train.shape[1]}")

# ---- Fuse features ----
print("[Step 5] Fusing features...")
X_train = np.concatenate([X_hjorth_train, X_ts_train], axis=1)
X_test = np.concatenate([X_hjorth_test, X_ts_test], axis=1)

print(f"\nFinal feature shapes:")
print(f"  Training: {X_train.shape}")
print(f"  Testing:  {X_test.shape}")
print(f"  Total features: {X_train.shape[1]} (Hjorth: {X_hjorth_train.shape[1]}, Riemannian: {X_ts_train.shape[1]})")

# ============================================================
# TRAIN XGBOOST
# ============================================================

print("\n" + "="*60)
print("TRAINING XGBOOST CLASSIFIER")
print("="*60)

model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

# ============================================================
# EVALUATION
# ============================================================

print("\n" + "="*60)
print("EVALUATION RESULTS")
print("="*60)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print(f"\nROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

# ============================================================
# FEATURE IMPORTANCE
# ============================================================
print("\n" + "="*60)
print("FEATURE IMPORTANCE")
print("="*60)

importances = model.feature_importances_
n_hjorth = X_hjorth_train.shape[1]
n_riemannian = X_ts_train.shape[1]

importance_hjorth = np.sum(importances[:n_hjorth])
importance_riemannian = np.sum(importances[n_hjorth:])

print(f"\n  Hjorth features ({n_hjorth}):     {importance_hjorth:.4f} ({100*importance_hjorth:.1f}%)")
print(f"  Riemannian features ({n_riemannian}): {importance_riemannian:.4f} ({100*importance_riemannian:.1f}%)")

# Top 10 features
top_indices = np.argsort(importances)[-10:][::-1]
print(f"\nTop 10 most important features:")
for rank, idx in enumerate(top_indices, 1):
    feat_type = "Hjorth" if idx < n_hjorth else "Riemannian"
    print(f"  {rank:2d}. {feat_type:10s} (index {idx:4d}): {importances[idx]:.6f}")

# fixed

In [1]:
import os
import numpy as np
import mne

ROOT = "/Users/roman/PycharmProjects/brain_data/ds005516"
WINDOW_SEC = 2.0
STEP_SEC = 2.0
TEST_RATIO = 0.3
SEED = 42
RESAMPLE_RATE = 125 # was 500

SUBJECTS = [
    "sub-NDARAB678VYW","sub-NDARAB683CYD","sub-NDARAC296UCB","sub-NDARAD459XJK",
    "sub-NDARAG429CGW","sub-NDARAG788YV9","sub-NDARAJ182PTB","sub-NDARAJ401AX3",
    "sub-NDARAJ674WJT","sub-NDARAK772VFJ","sub-NDARAM177DKJ","sub-NDARAM946HJE",
    "sub-NDARAN302EEM","sub-NDARAP283ZBW","sub-NDARAP748AWX","sub-NDARAU517MC6",
    "sub-NDARAW427GWK","sub-NDARAW620GJ8","sub-NDARAY761BFH","sub-NDARAY977BZT",
    "sub-NDARAZ532KK0","sub-NDARBB542URX","sub-NDARBE096YK6","sub-NDARBF402JLH",
    "sub-NDARBH275EXT","sub-NDARBH482JK1","sub-NDARBH789CUP","sub-NDARBK194FF5",
    "sub-NDARBP713ZWA","sub-NDARBT780KVC"
]

ACTIVE_TASKS = {"contrastChangeDetection", "symbolSearch"}
PASSIVE_TASKS = {
    "RestingState","surroundSupp","DespicableMe",
    "DiaryOfAWimpyKid","FunwithFractals","ThePresent"
}
ALL_TASKS = ACTIVE_TASKS | PASSIVE_TASKS


# ------------------------------------------------------------
# split subjects
# ------------------------------------------------------------
def split_subjects(subjects):
    subjects = np.array(subjects)
    rng = np.random.RandomState(SEED)
    rng.shuffle(subjects)

    n_test = int(len(subjects) * TEST_RATIO)
    return subjects[n_test:], subjects[:n_test]


# ------------------------------------------------------------
# file handling
# ------------------------------------------------------------
def list_eeg_files(subject):
    eeg_dir = os.path.join(ROOT, subject, "eeg")
    if not os.path.exists(eeg_dir):
        return []

    return [
        os.path.join(eeg_dir, f)
        for f in os.listdir(eeg_dir)
        if f.endswith("_eeg.set")
    ]


def parse_task(path):
    for part in os.path.basename(path).split("_"):
        if part.startswith("task-"):
            return part.replace("task-", "")
    return None


def select_run1(files):
    groups = {}

    for f in files:
        task = parse_task(f)
        if task:
            groups.setdefault(task, []).append(f)

    selected = []
    for task, g in groups.items():
        run1 = [x for x in g if "run-1" in x]
        selected.append(run1[0] if run1 else sorted(g)[0])

    return selected


def get_label(task):
    if task in ACTIVE_TASKS:
        return 1
    if task in PASSIVE_TASKS:
        return 0
    return None


# ------------------------------------------------------------
# window extraction (NO processing)
# ------------------------------------------------------------
def extract_windows(raw, label, max_windows=50):
    raw.load_data()
    raw.resample(RESAMPLE_RATE)

    sfreq = raw.info["sfreq"]
    win = int(WINDOW_SEC * sfreq)
    step = int(STEP_SEC * sfreq)

    data = raw.get_data()   # (channels, time)

    X, y = [], []

    count = 0

    for start in range(0, data.shape[1] - win, step):
        window = data[:, start:start + win]   # (channels, time)

        X.append(window)
        y.append(label)

        count += 1
        if count >= max_windows:
            break

    return X, y


# ------------------------------------------------------------
# dataset builder
# ------------------------------------------------------------
def build_dataset(subjects):
    X_all, y_all = [], []

    for sub in subjects:
        eeg_files = select_run1(list_eeg_files(sub))

        for path in eeg_files:
            task = parse_task(path)
            label = get_label(task)

            if task not in ALL_TASKS or label is None:
                continue

            try:
                raw = mne.io.read_raw_eeglab(path, preload=False)

                X, y = extract_windows(raw, label)

                X_all.extend(X)
                y_all.extend(y)

                del raw

            except Exception as e:
                print("Skip:", path, e)

    return np.array(X_all), np.array(y_all)


# ------------------------------------------------------------
# run
# ------------------------------------------------------------
train_subjects, test_subjects = split_subjects(SUBJECTS)

X_train, y_train = build_dataset(train_subjects)
X_test, y_test   = build_dataset(test_subjects)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-DespicableMe_eeg.set
Reading 0 ... 86139  =      0.000 ...   172.278 secs...


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_31693/1395971438.py:136: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=False)


Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-ThePresent_eeg.set
Reading 0 ... 102439  =      0.000 ...   204.878 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-FunwithFractals_eeg.set
Reading 0 ... 82352  =      0.000 ...   164.704 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-contrastChangeDetection_run-1_eeg.set


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_31693/1395971438.py:136: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=False)


Reading 0 ... 179228  =      0.000 ...   358.456 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-surroundSupp_run-1_eeg.set
Reading 0 ... 139316  =      0.000 ...   278.632 secs...


...

Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARBH275EXT/eeg/sub-NDARBH275EXT_task-DiaryOfAWimpyKid_eeg.set
Reading 0 ... 59521  =      0.000 ...   119.042 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARBH275EXT/eeg/sub-NDARBH275EXT_task-ThePresent_eeg.set
Reading 0 ... 102301  =      0.000 ...   204.602 secs...


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_31693/1395971438.py:136: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=False)


Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARBH275EXT/eeg/sub-NDARBH275EXT_task-DespicableMe_eeg.set
Reading 0 ... 86143  =      0.000 ...   172.286 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAN302EEM/eeg/sub-NDARAN302EEM_task-FunwithFractals_eeg.set
Reading 0 ... 82263  =      0.000 ...   164.526 secs...


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_31693/1395971438.py:136: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=False)
/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_31693/1395971438.py:136: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=False)


Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAN302EEM/eeg/sub-NDARAN302EEM_task-DespicableMe_eeg.set
Reading 0 ... 155653  =      0.000 ...   311.306 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAN302EEM/eeg/sub-NDARAN302EEM_task-ThePresent_eeg.set
Reading 0 ... 102809  =      0.000 ...   205.618 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAN302EEM/eeg/sub-NDARAN302EEM_task-DiaryOfAWimpyKid_eeg.set
Reading 0 ... 75297  =      0.000 ...   150.594 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAN302EEM/eeg/sub-NDARAN302EEM_task-surroundSupp_run-1_eeg.set
Reading 0 ... 175237  =      0.000 ...   350.474 secs...
Train: (6218, 129, 250) (6218,)
Test : (3247, 129, 250) (3247,)


In [4]:
X_train.shape

(6218, 129, 250)

## flat channels removal

In [5]:
X_train[0, :, :]

array([[ 0.19389378,  0.19389378,  0.19389378, ...,  0.19389378,
         0.19389378,  0.19389378],
       [-0.00702372, -0.00694516, -0.00695759, ..., -0.00697376,
        -0.00696397, -0.00696883],
       [-0.00419555, -0.00412375, -0.0041311 , ..., -0.00413019,
        -0.00412096, -0.00412845],
       ...,
       [-0.01175067, -0.01168705, -0.01167929, ..., -0.01167273,
        -0.01165791, -0.0116669 ],
       [ 0.00059902,  0.00065667,  0.00066121, ...,  0.00076926,
         0.00077553,  0.00076509],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ]], shape=(129, 250))

In [6]:
X_train[1, :, :]

array([[ 5.55111509e-09,  5.55111509e-09,  5.55111509e-09, ...,
         5.55111509e-09,  5.55111509e-09,  5.55111509e-09],
       [-6.43662816e-01, -7.60669241e-01, -4.94276393e-01, ...,
        -6.79072255e-01, -1.04217884e+00, -1.20030345e+00],
       [-6.23218629e-01, -6.14063801e-01, -1.72849868e-01, ...,
        -2.09638019e-01, -5.68514780e-01, -9.40936183e-01],
       ...,
       [-3.86415023e-01, -2.91014537e-01,  3.36834656e-02, ...,
         8.57980362e-01,  4.65034319e-01, -1.99160991e-01],
       [-1.49640305e+00, -1.42078512e+00, -7.11346000e-01, ...,
         1.31098961e+00,  6.57102387e-01, -3.56655723e-01],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00]],
      shape=(129, 1000))

In [6]:
import numpy as np

eps = 1e-10

for i in range(min(10, len(X_train))):
    stds = np.std(X_train[i], axis=1)   # per-channel std
    flat_idx = np.where(stds < eps)[0]

    print(f"Window {i}: {len(flat_idx)} flat channels")
    print(flat_idx)
    print("-" * 40)

Window 0: 6 flat channels
[  0   9  16  42 121 128]
----------------------------------------
Window 1: 6 flat channels
[  0   9  16  42 121 128]
----------------------------------------
Window 2: 6 flat channels
[  0   9  16  42 121 128]
----------------------------------------
Window 3: 6 flat channels
[  0   9  16  42 121 128]
----------------------------------------
Window 4: 6 flat channels
[  0   9  16  42 121 128]
----------------------------------------
Window 5: 6 flat channels
[  0   9  16  42 121 128]
----------------------------------------
Window 6: 6 flat channels
[  0   9  16  42 121 128]
----------------------------------------
Window 7: 6 flat channels
[  0   9  16  42 121 128]
----------------------------------------
Window 8: 6 flat channels
[  0   9  16  42 121 128]
----------------------------------------
Window 9: 6 flat channels
[  0   9  16  42 121 128]
----------------------------------------


In [7]:
import numpy as np

eps = 1e-8
flat_idx_common = set()

for i in range(len(X_train)):
    stds = np.std(X_train[i], axis=1)
    flat_idx = set(np.where(stds < eps)[0])

    flat_idx_common |= flat_idx

print(sorted(flat_idx_common))

[np.int64(0), np.int64(7), np.int64(9), np.int64(10), np.int64(16), np.int64(20), np.int64(24), np.int64(28), np.int64(31), np.int64(42), np.int64(55), np.int64(89), np.int64(105), np.int64(106), np.int64(118), np.int64(121), np.int64(126), np.int64(128)]


In [8]:
import numpy as np

eps = 1e-8
flat_idx_common_test = set()

for i in range(len(X_test)):
    stds = np.std(X_test[i], axis=1)
    flat_idx = set(np.where(stds < eps)[0])

    flat_idx_common_test |= flat_idx

print(sorted(flat_idx_common_test))

[np.int64(2), np.int64(3), np.int64(9), np.int64(16), np.int64(42), np.int64(44), np.int64(47), np.int64(48), np.int64(55), np.int64(79), np.int64(89), np.int64(106), np.int64(112), np.int64(118), np.int64(121), np.int64(125), np.int64(128)]


In [9]:
flat_idx_common - flat_idx_common_test

{np.int64(0),
 np.int64(7),
 np.int64(10),
 np.int64(20),
 np.int64(24),
 np.int64(28),
 np.int64(31),
 np.int64(105),
 np.int64(126)}

In [10]:
flat_idx_common_test - flat_idx_common

{np.int64(2),
 np.int64(3),
 np.int64(44),
 np.int64(47),
 np.int64(48),
 np.int64(79),
 np.int64(112),
 np.int64(125)}

In [11]:
flat_idx_common_overall = flat_idx_common | flat_idx_common_test

In [12]:
bad_channels = np.array(sorted(flat_idx_common_overall))

# build boolean mask: True = keep
n_channels = X_train.shape[1]
keep_mask = np.ones(n_channels, dtype=bool)
keep_mask[bad_channels] = False

# apply to both splits
X_train_fixed = X_train[:, keep_mask, :]
X_test_fixed  = X_test[:,  keep_mask, :]

print("Original shape:", X_train.shape)
print("Fixed shape   :", X_train_fixed.shape)

Original shape: (6218, 129, 250)
Fixed shape   : (6218, 103, 250)


## pipeline only for hjorth features

In [13]:
import numpy as np
import mne

from scipy.signal import welch
from scipy.stats import entropy
from itertools import combinations

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.covariance import LedoitWolf

from xgboost import XGBClassifier

from pyriemann.utils.mean import mean_covariance
from pyriemann.tangentspace import TangentSpace
from scipy.linalg import fractional_matrix_power

def compute_covariance(w):
    w = w - np.mean(w, axis=1, keepdims=True)

    s = w.shape[1]
    C = (w @ w.T) / (s - 1)

    # small ridge regularization (NOT full Ledoit collapse)
    C += REG * np.eye(C.shape[0])

    return C

def hjorth_activity(x): return np.var(x)

def hjorth_mobility(x):
    v = np.var(x)
    return np.sqrt(np.var(np.diff(x)) / (v + 1e-12))

def hjorth_complexity(x):
    m = hjorth_mobility(x)
    return hjorth_mobility(np.diff(x)) / (m + 1e-12)

def extract_features_window(data, ch_names):
    feats = {}

    for i, ch in enumerate(ch_names):
        x = data[i]

        feats[f"{ch}_a"] = hjorth_activity(x)
        feats[f"{ch}_m"] = hjorth_mobility(x)
        feats[f"{ch}_c"] = hjorth_complexity(x)

        feats[f"{ch}_log_a"] = np.log(feats[f"{ch}_a"] + 1e-12)
        feats[f"{ch}_log_m"] = np.log(feats[f"{ch}_m"] + 1e-12)

    return feats

def whiten(C, M):
    M_inv_sqrt = fractional_matrix_power(M, -0.5)
    return M_inv_sqrt @ C @ M_inv_sqrt

# ============================================================
# FEATURE EXTRACTION FROM PRECOMPUTED WINDOWS
# ============================================================

def extract_feature_matrix(X):
    """
    X: (n_samples, n_channels, n_times)
    returns: (n_samples, n_features)
    """
    all_feats = []

    for w in X:

        feats = extract_features_window(w, [f"ch{i}" for i in range(w.shape[0])])
        vals = np.array(list(feats.values()))

        if not np.all(np.isfinite(vals)):
            continue

        all_feats.append(vals)

    return np.array(all_feats)


# compute features
X_train_feats = extract_feature_matrix(X_train_fixed)
X_test_feats  = extract_feature_matrix(X_test_fixed)

print("Feature shapes:")
print("Train:", X_train_feats.shape)
print("Test :", X_test_feats.shape)

Feature shapes:
Train: (6218, 515)
Test : (3247, 515)


In [15]:
y_train.mean(), y_test.mean()

(np.float64(0.1929880990672242), np.float64(0.21558361564521097))

In [14]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42
)

model.fit(X_train_feats, y_train)

y_pred = model.predict(X_test_feats)
y_prob = model.predict_proba(X_test_feats)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.78      1.00      0.88      2547
           1       0.00      0.00      0.00       700

    accuracy                           0.78      3247
   macro avg       0.39      0.50      0.44      3247
weighted avg       0.62      0.78      0.69      3247

ROC-AUC: 0.5204383308093556


## with cov

In [17]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

from pyriemann.utils.mean import mean_covariance
from pyriemann.tangentspace import TangentSpace
from scipy.linalg import fractional_matrix_power

REG = 1e-5

# ============================================================
# 1. COVARIANCE FROM WINDOW SIGNALS (IMPORTANT PART)
# ============================================================

def compute_covariance(w):
    # w: (channels, time)
    w = w - np.mean(w, axis=1, keepdims=True)

    s = w.shape[1]
    C = (w @ w.T) / (s - 1)

    C += REG * np.eye(C.shape[0])
    return C


# ============================================================
# 2. BUILD COVARIANCES FROM FIXED WINDOWS
# ============================================================

def build_covariances(X):
    covs = []

    for w in X:
        # w: (channels, time)
        if np.any(np.std(w, axis=1) < 1e-10):
            continue

        C = compute_covariance(w)

        if np.any(~np.isfinite(C)):
            continue

        covs.append(C)

    return np.array(covs)


# ============================================================
# 3. BUILD DATASETS
# ============================================================

covs_train = build_covariances(X_train_fixed)
covs_test  = build_covariances(X_test_fixed)

y_train = y_train[:len(covs_train)]
y_test  = y_test[:len(covs_test)]


# ============================================================
# 4. RIEMANNIAN MEAN + WHITENING
# ============================================================

M = mean_covariance(covs_train, metric='riemann')

def whiten(C, M):
    M_inv_sqrt = fractional_matrix_power(M, -0.5)
    return M_inv_sqrt @ C @ M_inv_sqrt


covs_train_w = np.array([whiten(C, M) for C in covs_train])
covs_test_w  = np.array([whiten(C, M) for C in covs_test])


# ============================================================
# 5. TANGENT SPACE EMBEDDING
# ============================================================

ts = TangentSpace(metric='riemann')
ts.fit(covs_train_w)

X_ts_train = ts.transform(covs_train_w)
X_ts_test  = ts.transform(covs_test_w)


# ============================================================
# 6. MODEL
# ============================================================

model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42
)

model.fit(X_ts_train, y_train)

y_pred = model.predict(X_ts_test)
y_prob = model.predict_proba(X_ts_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.78      1.00      0.88      2547
           1       0.00      0.00      0.00       700

    accuracy                           0.78      3247
   macro avg       0.39      0.50      0.44      3247
weighted avg       0.61      0.78      0.69      3247

ROC-AUC: 0.49613382691121205


# change to resting state as just rest

In [9]:
import os
import numpy as np
import mne

ROOT = "/Users/roman/PycharmProjects/brain_data/ds005516"
WINDOW_SEC = 2.0
STEP_SEC = 2.0
TEST_RATIO = 0.3
SEED = 42
RESAMPLE_RATE = 125 # was 500

SUBJECTS = [
    "sub-NDARAB678VYW","sub-NDARAB683CYD","sub-NDARAC296UCB","sub-NDARAD459XJK",
    "sub-NDARAG429CGW","sub-NDARAG788YV9","sub-NDARAJ182PTB","sub-NDARAJ401AX3",
    "sub-NDARAJ674WJT","sub-NDARAK772VFJ","sub-NDARAM177DKJ","sub-NDARAM946HJE",
    "sub-NDARAN302EEM","sub-NDARAP283ZBW","sub-NDARAP748AWX","sub-NDARAU517MC6",
    "sub-NDARAW427GWK","sub-NDARAW620GJ8","sub-NDARAY761BFH","sub-NDARAY977BZT",
    "sub-NDARAZ532KK0","sub-NDARBB542URX","sub-NDARBE096YK6","sub-NDARBF402JLH",
    "sub-NDARBH275EXT","sub-NDARBH482JK1","sub-NDARBH789CUP","sub-NDARBK194FF5",
    "sub-NDARBP713ZWA","sub-NDARBT780KVC"
]

# ACTIVE_TASKS = {"contrastChangeDetection", "symbolSearch", "surroundSupp","DespicableMe",
#     "DiaryOfAWimpyKid","FunwithFractals","ThePresent"}
ACTIVE_TASKS = {"contrastChangeDetection", "symbolSearch"}
PASSIVE_TASKS = {
    "RestingState"
}
ALL_TASKS = ACTIVE_TASKS | PASSIVE_TASKS


# ------------------------------------------------------------
# split subjects
# ------------------------------------------------------------
def split_subjects(subjects):
    subjects = np.array(subjects)
    rng = np.random.RandomState(SEED)
    rng.shuffle(subjects)

    n_test = int(len(subjects) * TEST_RATIO)
    return subjects[n_test:], subjects[:n_test]


# ------------------------------------------------------------
# file handling
# ------------------------------------------------------------
def list_eeg_files(subject):
    eeg_dir = os.path.join(ROOT, subject, "eeg")
    if not os.path.exists(eeg_dir):
        return []

    return [
        os.path.join(eeg_dir, f)
        for f in os.listdir(eeg_dir)
        if f.endswith("_eeg.set")
    ]


def parse_task(path):
    for part in os.path.basename(path).split("_"):
        if part.startswith("task-"):
            return part.replace("task-", "")
    return None


def select_run1(files):
    groups = {}

    for f in files:
        task = parse_task(f)
        if task:
            groups.setdefault(task, []).append(f)

    selected = []
    for task, g in groups.items():
        run1 = [x for x in g if "run-1" in x]
        selected.append(run1[0] if run1 else sorted(g)[0])

    return selected


def get_label(task):
    if task in ACTIVE_TASKS:
        return 1
    if task in PASSIVE_TASKS:
        return 0
    return None


# ------------------------------------------------------------
# window extraction (NO processing)
# ------------------------------------------------------------
def extract_windows(raw, label, max_windows=50):
    raw.load_data()
    raw.resample(RESAMPLE_RATE)

    sfreq = raw.info["sfreq"]
    win = int(WINDOW_SEC * sfreq)
    step = int(STEP_SEC * sfreq)

    data = raw.get_data()   # (channels, time)

    X, y = [], []

    count = 0

    for start in range(0, data.shape[1] - win, step):
        window = data[:, start:start + win]   # (channels, time)

        X.append(window)
        y.append(label)

        count += 1
        if count >= max_windows:
            break

    return X, y


# ------------------------------------------------------------
# dataset builder
# ------------------------------------------------------------
def build_dataset(subjects):
    X_all, y_all = [], []

    for sub in subjects:
        eeg_files = select_run1(list_eeg_files(sub))

        for path in eeg_files:
            task = parse_task(path)
            label = get_label(task)

            if task not in ALL_TASKS or label is None:
                continue

            try:
                raw = mne.io.read_raw_eeglab(path, preload=False)

                X, y = extract_windows(raw, label)

                X_all.extend(X)
                y_all.extend(y)

                del raw

            except Exception as e:
                print("Skip:", path, e)

    return np.array(X_all), np.array(y_all)


# ------------------------------------------------------------
# run
# ------------------------------------------------------------
train_subjects, test_subjects = split_subjects(SUBJECTS)

X_train, y_train = build_dataset(train_subjects)
X_test, y_test   = build_dataset(test_subjects)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-DespicableMe_eeg.set
Reading 0 ... 86139  =      0.000 ...   172.278 secs...


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_31693/35044500.py:136: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=False)


Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-ThePresent_eeg.set
Reading 0 ... 102439  =      0.000 ...   204.878 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-FunwithFractals_eeg.set
Reading 0 ... 82352  =      0.000 ...   164.704 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-contrastChangeDetection_run-1_eeg.set


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_31693/35044500.py:136: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=False)


Reading 0 ... 179228  =      0.000 ...   358.456 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-surroundSupp_run-1_eeg.set
Reading 0 ... 139316  =      0.000 ...   278.632 secs...


...

Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARBH275EXT/eeg/sub-NDARBH275EXT_task-DiaryOfAWimpyKid_eeg.set
Reading 0 ... 59521  =      0.000 ...   119.042 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARBH275EXT/eeg/sub-NDARBH275EXT_task-ThePresent_eeg.set
Reading 0 ... 102301  =      0.000 ...   204.602 secs...


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_31693/35044500.py:136: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=False)


Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARBH275EXT/eeg/sub-NDARBH275EXT_task-DespicableMe_eeg.set
Reading 0 ... 86143  =      0.000 ...   172.286 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAN302EEM/eeg/sub-NDARAN302EEM_task-FunwithFractals_eeg.set
Reading 0 ... 82263  =      0.000 ...   164.526 secs...


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_31693/35044500.py:136: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=False)
/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_31693/35044500.py:136: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=False)


Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAN302EEM/eeg/sub-NDARAN302EEM_task-DespicableMe_eeg.set
Reading 0 ... 155653  =      0.000 ...   311.306 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAN302EEM/eeg/sub-NDARAN302EEM_task-ThePresent_eeg.set
Reading 0 ... 102809  =      0.000 ...   205.618 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAN302EEM/eeg/sub-NDARAN302EEM_task-DiaryOfAWimpyKid_eeg.set
Reading 0 ... 75297  =      0.000 ...   150.594 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAN302EEM/eeg/sub-NDARAN302EEM_task-surroundSupp_run-1_eeg.set
Reading 0 ... 175237  =      0.000 ...   350.474 secs...
Train: (6218, 129, 250) (6218,)
Test : (3247, 129, 250) (3247,)


## flat channel removal

In [10]:
import numpy as np

eps = 1e-8
flat_idx_common = set()

for i in range(len(X_train)):
    stds = np.std(X_train[i], axis=1)
    flat_idx = set(np.where(stds < eps)[0])

    flat_idx_common |= flat_idx

print(sorted(flat_idx_common))

[np.int64(0), np.int64(7), np.int64(9), np.int64(10), np.int64(16), np.int64(20), np.int64(24), np.int64(28), np.int64(31), np.int64(42), np.int64(55), np.int64(89), np.int64(105), np.int64(106), np.int64(118), np.int64(121), np.int64(126), np.int64(128)]


In [11]:
import numpy as np

eps = 1e-8
flat_idx_common_test = set()

for i in range(len(X_test)):
    stds = np.std(X_test[i], axis=1)
    flat_idx = set(np.where(stds < eps)[0])

    flat_idx_common_test |= flat_idx

print(sorted(flat_idx_common_test))

[np.int64(2), np.int64(3), np.int64(9), np.int64(16), np.int64(42), np.int64(44), np.int64(47), np.int64(48), np.int64(55), np.int64(79), np.int64(89), np.int64(106), np.int64(112), np.int64(118), np.int64(121), np.int64(125), np.int64(128)]


In [12]:
bad_channels = np.array(sorted(flat_idx_common_test | flat_idx_common))

# build boolean mask: True = keep
n_channels = X_train.shape[1]
keep_mask = np.ones(n_channels, dtype=bool)
keep_mask[bad_channels] = False

# apply to both splits
X_train_fixed = X_train[:, keep_mask, :]
X_test_fixed  = X_test[:,  keep_mask, :]

print("Original shape:", X_train.shape)
print("Fixed shape   :", X_train_fixed.shape)

Original shape: (6218, 129, 250)
Fixed shape   : (6218, 103, 250)


## hjorth

In [23]:
import numpy as np
import mne

from scipy.signal import welch
from scipy.stats import entropy
from itertools import combinations

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.covariance import LedoitWolf

from xgboost import XGBClassifier

from pyriemann.utils.mean import mean_covariance
from pyriemann.tangentspace import TangentSpace
from scipy.linalg import fractional_matrix_power

def compute_covariance(w):
    w = w - np.mean(w, axis=1, keepdims=True)

    s = w.shape[1]
    C = (w @ w.T) / (s - 1)

    # small ridge regularization (NOT full Ledoit collapse)
    C += REG * np.eye(C.shape[0])

    return C

def hjorth_activity(x): return np.var(x)

def hjorth_mobility(x):
    v = np.var(x)
    return np.sqrt(np.var(np.diff(x)) / (v + 1e-12))

def hjorth_complexity(x):
    m = hjorth_mobility(x)
    return hjorth_mobility(np.diff(x)) / (m + 1e-12)

def extract_features_window(data, ch_names):
    feats = {}

    for i, ch in enumerate(ch_names):
        x = data[i]

        feats[f"{ch}_a"] = hjorth_activity(x)
        feats[f"{ch}_m"] = hjorth_mobility(x)
        feats[f"{ch}_c"] = hjorth_complexity(x)

        feats[f"{ch}_log_a"] = np.log(feats[f"{ch}_a"] + 1e-12)
        feats[f"{ch}_log_m"] = np.log(feats[f"{ch}_m"] + 1e-12)

    return feats

def whiten(C, M):
    M_inv_sqrt = fractional_matrix_power(M, -0.5)
    return M_inv_sqrt @ C @ M_inv_sqrt

# ============================================================
# FEATURE EXTRACTION FROM PRECOMPUTED WINDOWS
# ============================================================

def extract_feature_matrix(X):
    """
    X: (n_samples, n_channels, n_times)
    returns: (n_samples, n_features)
    """
    all_feats = []

    for w in X:

        feats = extract_features_window(w, [f"ch{i}" for i in range(w.shape[0])])
        vals = np.array(list(feats.values()))

        if not np.all(np.isfinite(vals)):
            continue

        all_feats.append(vals)

    return np.array(all_feats)


# compute features
X_train_feats = extract_feature_matrix(X_train_fixed)
X_test_feats  = extract_feature_matrix(X_test_fixed)

print("Feature shapes:")
print("Train:", X_train_feats.shape)
print("Test :", X_test_feats.shape)

model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42
)

model.fit(X_train_feats, y_train)

y_pred = model.predict(X_test_feats)
y_prob = model.predict_proba(X_test_feats)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Feature shapes:
Train: (6218, 515)
Test : (3247, 515)
              precision    recall  f1-score   support

           0       0.04      0.01      0.01       347
           1       0.89      0.98      0.93      2900

    accuracy                           0.88      3247
   macro avg       0.46      0.49      0.47      3247
weighted avg       0.80      0.88      0.84      3247

ROC-AUC: 0.6434452946437443


## cov

In [22]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

from pyriemann.utils.mean import mean_covariance
from pyriemann.tangentspace import TangentSpace
from scipy.linalg import fractional_matrix_power

REG = 1e-5

# ============================================================
# 1. COVARIANCE FROM WINDOW SIGNALS (IMPORTANT PART)
# ============================================================

def compute_covariance(w):
    # w: (channels, time)
    w = w - np.mean(w, axis=1, keepdims=True)

    s = w.shape[1]
    C = (w @ w.T) / (s - 1)

    C += REG * np.eye(C.shape[0])
    return C


# ============================================================
# 2. BUILD COVARIANCES FROM FIXED WINDOWS
# ============================================================

def build_covariances(X):
    covs = []

    for w in X:
        # w: (channels, time)
        if np.any(np.std(w, axis=1) < 1e-10):
            continue

        C = compute_covariance(w)

        if np.any(~np.isfinite(C)):
            continue

        covs.append(C)

    return np.array(covs)


# ============================================================
# 3. BUILD DATASETS
# ============================================================

covs_train = build_covariances(X_train_fixed)
covs_test  = build_covariances(X_test_fixed)

y_train = y_train[:len(covs_train)]
y_test  = y_test[:len(covs_test)]


# ============================================================
# 4. RIEMANNIAN MEAN + WHITENING
# ============================================================

M = mean_covariance(covs_train, metric='riemann')

def whiten(C, M):
    M_inv_sqrt = fractional_matrix_power(M, -0.5)
    return M_inv_sqrt @ C @ M_inv_sqrt


covs_train_w = np.array([whiten(C, M) for C in covs_train])
covs_test_w  = np.array([whiten(C, M) for C in covs_test])


# ============================================================
# 5. TANGENT SPACE EMBEDDING
# ============================================================

ts = TangentSpace(metric='riemann')
ts.fit(covs_train_w)

X_ts_train = ts.transform(covs_train_w)
X_ts_test  = ts.transform(covs_test_w)


# ============================================================
# 6. MODEL
# ============================================================

model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42
)

model.fit(X_ts_train, y_train)

y_pred = model.predict(X_ts_test)
y_prob = model.predict_proba(X_ts_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.19      0.05      0.08       347
           1       0.90      0.97      0.93      2900

    accuracy                           0.88      3247
   macro avg       0.54      0.51      0.51      3247
weighted avg       0.82      0.88      0.84      3247

ROC-AUC: 0.6511428003577462


## combined

In [24]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

from pyriemann.utils.mean import mean_covariance
from pyriemann.tangentspace import TangentSpace
from scipy.linalg import fractional_matrix_power

REG = 1e-5

# ============================================================
# 1. HJORTH FEATURES (per window)
# ============================================================

def hjorth_activity(x):
    return np.var(x)

def hjorth_mobility(x):
    v = np.var(x)
    return np.sqrt(np.var(np.diff(x)) / (v + 1e-12))

def hjorth_complexity(x):
    m = hjorth_mobility(x)
    return hjorth_mobility(np.diff(x)) / (m + 1e-12)


def extract_hjorth(X):
    """
    X: (n_samples, n_channels, n_times)
    returns: (n_samples, n_channels * 5 features)
    """
    feats_all = []

    for w in X:
        feats = []

        for ch in range(w.shape[0]):
            x = w[ch]

            a = hjorth_activity(x)
            m = hjorth_mobility(x)
            c = hjorth_complexity(x)

            feats.extend([
                a,
                m,
                c,
                np.log(a + 1e-12),
                np.log(m + 1e-12)
            ])

        feats_all.append(feats)

    return np.array(feats_all)


# ============================================================
# 2. COVARIANCE FEATURES
# ============================================================

def compute_covariance(w):
    w = w - np.mean(w, axis=1, keepdims=True)
    s = w.shape[1]

    C = (w @ w.T) / (s - 1)
    C += REG * np.eye(C.shape[0])

    return C


def build_covs(X):
    covs = []

    for w in X:
        if np.any(np.std(w, axis=1) < 1e-10):
            continue
        covs.append(compute_covariance(w))

    return np.array(covs)


def whiten(C, M):
    M_inv_sqrt = fractional_matrix_power(M, -0.5)
    return M_inv_sqrt @ C @ M_inv_sqrt


# ============================================================
# 3. HJORTH FEATURES
# ============================================================

X_h_train = extract_hjorth(X_train_fixed)
X_h_test  = extract_hjorth(X_test_fixed)

scaler = StandardScaler()
X_h_train = scaler.fit_transform(X_h_train)
X_h_test  = scaler.transform(X_h_test)


# ============================================================
# 4. RIEMANNIAN FEATURES
# ============================================================

covs_train = build_covs(X_train_fixed)
covs_test  = build_covs(X_test_fixed)

# IMPORTANT: ensure alignment (same number of samples)
n = min(len(covs_train), len(covs_test))
covs_train = covs_train[:n]
covs_test  = covs_test[:n]

y_train = y_train[:n]
y_test  = y_test[:n]

# Riemann mean
M = mean_covariance(covs_train, metric='riemann')

covs_train_w = np.array([whiten(C, M) for C in covs_train])
covs_test_w  = np.array([whiten(C, M) for C in covs_test])

ts = TangentSpace(metric='riemann')
ts.fit(covs_train_w)

X_ts_train = ts.transform(covs_train_w)
X_ts_test  = ts.transform(covs_test_w)


# ============================================================
# 5. FUSION (CRITICAL STEP)
# ============================================================

X_train = np.concatenate([X_h_train[:n], X_ts_train], axis=1)
X_test  = np.concatenate([X_h_test[:n],  X_ts_test], axis=1)


# ============================================================
# 6. MODEL
# ============================================================

model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       347
           1       0.89      0.98      0.93      2900

    accuracy                           0.87      3247
   macro avg       0.45      0.49      0.47      3247
weighted avg       0.80      0.87      0.83      3247

ROC-AUC: 0.5976239689953294


## combined with more time features

In [25]:
import numpy as np
from scipy.signal import welch
from scipy.stats import entropy


def bandpower(psd, freqs, band):
    fmin, fmax = band
    mask = (freqs >= fmin) & (freqs <= fmax)
    return np.trapz(psd[mask], freqs[mask])


def spectral_entropy(psd):
    psd = np.nan_to_num(psd)
    total = np.sum(psd)
    if total == 0:
        return 0.0
    p = psd / total
    return entropy(p)


def spectral_complexity(psd):
    psd = np.nan_to_num(psd)
    total = np.sum(psd)
    if total == 0:
        return 0.0

    p = psd / total
    uniform = np.ones_like(p) / len(p)
    return entropy(p, uniform)

In [26]:
BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta":  (13, 30),
    "gamma": (30, 45)
}


def extract_spectral_features(X, sfreq=125):
    """
    X: (n_samples, n_channels, n_times)
    returns: (n_samples, n_channels * spectral_features)
    """

    all_feats = []

    for w in X:
        feats = []

        for ch in range(w.shape[0]):
            x = w[ch]

            freqs, psd = welch(x, fs=sfreq, nperseg=min(len(x), 256))

            # bandpowers
            bp = []
            for band in BANDS.values():
                bp.append(bandpower(psd, freqs, band))

            se = spectral_entropy(psd)
            sc = spectral_complexity(psd)

            feats.extend(bp + [se, sc])

        all_feats.append(feats)

    return np.array(all_feats)

In [27]:
# ============================================================
# HJORTH
# ============================================================

X_h_train = extract_hjorth(X_train_fixed)
X_h_test  = extract_hjorth(X_test_fixed)

from sklearn.preprocessing import StandardScaler
scaler_h = StandardScaler()

X_h_train = scaler_h.fit_transform(X_h_train)
X_h_test  = scaler_h.transform(X_h_test)


# ============================================================
# SPECTRAL
# ============================================================

X_s_train = extract_spectral_features(X_train_fixed, sfreq=125)
X_s_test  = extract_spectral_features(X_test_fixed, sfreq=125)

scaler_s = StandardScaler()

X_s_train = scaler_s.fit_transform(X_s_train)
X_s_test  = scaler_s.transform(X_s_test)


# ============================================================
# COVARIANCE + RIEMANN
# ============================================================

def compute_covariance(w):
    w = w - np.mean(w, axis=1, keepdims=True)
    C = (w @ w.T) / (w.shape[1] - 1)
    C += 1e-5 * np.eye(C.shape[0])
    return C


def build_covs(X):
    covs = []
    for w in X:
        if np.any(np.std(w, axis=1) < 1e-10):
            continue
        covs.append(compute_covariance(w))
    return np.array(covs)


def whiten(C, M):
    from scipy.linalg import fractional_matrix_power
    M_inv = fractional_matrix_power(M, -0.5)
    return M_inv @ C @ M_inv


covs_train = build_covs(X_train_fixed)
covs_test  = build_covs(X_test_fixed)

n = min(len(covs_train), len(covs_test))

covs_train = covs_train[:n]
covs_test  = covs_test[:n]

y_train = y_train[:n]
y_test  = y_test[:n]


from pyriemann.utils.mean import mean_covariance
from pyriemann.tangentspace import TangentSpace

M = mean_covariance(covs_train, metric='riemann')

covs_train_w = np.array([whiten(C, M) for C in covs_train])
covs_test_w  = np.array([whiten(C, M) for C in covs_test])

ts = TangentSpace(metric='riemann')
ts.fit(covs_train_w)

X_ts_train = ts.transform(covs_train_w)
X_ts_test  = ts.transform(covs_test_w)


# ============================================================
# FUSION
# ============================================================

X_train = np.concatenate([
    X_h_train[:n],
    X_s_train[:n],
    X_ts_train
], axis=1)

X_test = np.concatenate([
    X_h_test[:n],
    X_s_test[:n],
    X_ts_test
], axis=1)

/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_47999/2059797682.py:9: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(psd[mask], freqs[mask])


In [29]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.01      0.00      0.00       347
           1       0.89      0.98      0.93      2900

    accuracy                           0.87      3247
   macro avg       0.45      0.49      0.47      3247
weighted avg       0.80      0.87      0.83      3247

ROC-AUC: 0.574570704561264


# subjects info

In [31]:
SUBJECTS = [
    "sub-NDARAB678VYW","sub-NDARAB683CYD","sub-NDARAC296UCB","sub-NDARAD459XJK",
    "sub-NDARAG429CGW","sub-NDARAG788YV9","sub-NDARAJ182PTB","sub-NDARAJ401AX3",
    "sub-NDARAJ674WJT","sub-NDARAK772VFJ","sub-NDARAM177DKJ","sub-NDARAM946HJE",
    "sub-NDARAN302EEM","sub-NDARAP283ZBW","sub-NDARAP748AWX","sub-NDARAU517MC6",
    "sub-NDARAW427GWK","sub-NDARAW620GJ8","sub-NDARAY761BFH","sub-NDARAY977BZT",
    "sub-NDARAZ532KK0","sub-NDARBB542URX","sub-NDARBE096YK6","sub-NDARBF402JLH",
    "sub-NDARBH275EXT","sub-NDARBH482JK1","sub-NDARBH789CUP","sub-NDARBK194FF5",
    "sub-NDARBP713ZWA","sub-NDARBT780KVC"
]

In [32]:
import pandas as pd

df = pd.read_csv('../ds005516/participants.tsv', sep='\t')

filtered_df = df[df['participant_id'].isin(SUBJECTS)]

result = filtered_df[['participant_id', 'sex', 'age']]
result

,participant_id,sex,age
0,sub-NDARAB678VYW,M,20.1817
1,sub-NDARAB683CYD,M,7.2728
2,sub-NDARAC296UCB,M,22.0002
3,sub-NDARAD459XJK,F,7.3384
4,sub-NDARAG429CGW,M,7.4452
5,sub-NDARAG788YV9,F,15.0017
6,sub-NDARAJ182PTB,F,14.5364
7,sub-NDARAJ401AX3,M,9.0661
8,sub-NDARAJ674WJT,M,15.2786
9,sub-NDARAK772VFJ,M,9.8112


## median age, our

In [34]:
result['age'].median(), result['age'].mean()

(np.float64(9.056550000000001), np.float64(10.848696666666664))

## median age, overall

In [36]:
df['age'].median(), df['age'].mean()

(np.float64(9.71385), np.float64(10.39089069767442))

## proportion in gender, our

In [35]:
(result['sex'] == 'M').mean()

np.float64(0.6666666666666666)

## proportion in gender, overall

In [37]:
(df['sex'] == 'M').mean()

np.float64(0.6116279069767442)

# tasks info

In [40]:
filtered_df.columns

Index(['participant_id', 'release_number', 'sex', 'age', 'ehq_total',
       'commercial_use', 'full_pheno', 'p_factor', 'attention',
       'internalizing', 'externalizing', 'RestingState', 'DespicableMe',
       'FunwithFractals', 'ThePresent', 'DiaryOfAWimpyKid',
       'contrastChangeDetection_1', 'contrastChangeDetection_2',
       'contrastChangeDetection_3', 'surroundSupp_1', 'surroundSupp_2',
       'seqLearning6target', 'seqLearning8target', 'symbolSearch'],
      dtype='str')

In [41]:
tasks = ["contrastChangeDetection", "symbolSearch",
    "RestingState","surroundSupp","DespicableMe",
    "DiaryOfAWimpyKid","FunwithFractals","ThePresent"
]

filtered_df[['RestingState', 'DespicableMe',
       'FunwithFractals', 'ThePresent', 'DiaryOfAWimpyKid',
       'contrastChangeDetection_1', 'contrastChangeDetection_2',
       'contrastChangeDetection_3', 'surroundSupp_1', 'surroundSupp_2',
       'seqLearning6target', 'seqLearning8target']]

,RestingState,DespicableMe,FunwithFractals,ThePresent,DiaryOfAWimpyKid,contrastChangeDetection_1,contrastChangeDetection_2,contrastChangeDetection_3,surroundSupp_1,surroundSupp_2,seqLearning6target,seqLearning8target
0,available,available,available,available,available,available,available,available,available,available,unavailable,unavailable
1,unavailable,caution,caution,caution,caution,caution,unavailable,unavailable,caution,unavailable,unavailable,unavailable
2,caution,caution,unavailable,unavailable,caution,caution,caution,unavailable,caution,unavailable,unavailable,unavailable
3,caution,unavailable,unavailable,unavailable,unavailable,unavailable,unavailable,unavailable,unavailable,unavailable,unavailable,unavailable
4,available,unavailable,unavailable,unavailable,unavailable,unavailable,unavailable,unavailable,available,unavailable,unavailable,unavailable
5,available,available,available,available,available,available,available,available,available,available,unavailable,unavailable
6,unavailable,available,available,available,available,unavailable,unavailable,unavailable,unavailable,unavailable,unavailable,unavailable
7,unavailable,caution,caution,caution,caution,caution,caution,caution,caution,caution,unavailable,unavailable
8,caution,caution,caution,caution,caution,caution,caution,unavailable,caution,caution,unavailable,unavailable
9,caution,caution,caution,caution,caution,caution,caution,caution,caution,caution,unavailable,unavailable


In [48]:
import pandas as pd

# Assuming filtered_df already contains your subjects and the task columns
# Your original filtered_df from previous step

# List of task columns
task_columns = ['RestingState', 'DespicableMe', 'FunwithFractals', 'ThePresent',
                'DiaryOfAWimpyKid', 'contrastChangeDetection_1', 'contrastChangeDetection_2',
                'contrastChangeDetection_3', 'surroundSupp_1', 'surroundSupp_2',
                'seqLearning6target', 'seqLearning8target', 'symbolSearch']

# Create a copy to avoid modifying original
df_tasks = filtered_df.copy()

# Map: if value is 'unavailable' -> 0, else -> 1
for col in task_columns:
    df_tasks[col + '_mapped'] = (df_tasks[col] != 'unavailable').astype(int)

# Alternative more explicit version:
# for col in task_columns:
#     df_tasks[col + '_mapped'] = df_tasks[col].apply(lambda x: 0 if x == 'unavailable' else 1)

# Calculate for each subject:
# - Whether they have all 1's
# - Percentage of 1's
df_tasks['all_ones'] = df_tasks[[col + '_mapped' for col in task_columns]].all(axis=1)
df_tasks['percent_ones'] = df_tasks[[col + '_mapped' for col in task_columns]].mean(axis=1)

# Select subjects meeting criteria: all columns = 1 OR at least 80% = 1
selected_subjects = df_tasks[(df_tasks['all_ones'] == True) | (df_tasks['percent_ones'] >= 0.8)]

# Alternative if you want both conditions rigorously:
# selected_subjects = df_tasks[(df_tasks['all_ones'] == True) | (df_tasks['percent_ones'] >= 0.8)]

# View results
print(f"Total subjects in filtered_df: {len(df_tasks)}")
print(f"Subjects meeting criteria: {len(selected_subjects)}")
print(f"\nSelection criteria breakdown:")
print(f"  - Subjects with all 1's: {df_tasks['all_ones'].sum()}")
print(f"  - Subjects with >=80% 1's: {(df_tasks['percent_ones'] >= 0.8).sum()}")

# Display selected subjects with their original IDs and percentages
result_display = selected_subjects[['participant_id', 'percent_ones', 'all_ones']]
print(f"\nSelected subjects:")
print(result_display.to_string(index=False))

# If you want to add back the original task columns for verification
result_full = selected_subjects[['participant_id'] + task_columns + ['percent_ones', 'all_ones']]

Total subjects in filtered_df: 30
Subjects meeting criteria: 11

Selection criteria breakdown:
  - Subjects with all 1's: 0
  - Subjects with >=80% 1's: 11

Selected subjects:
  participant_id  percent_ones  all_ones
sub-NDARAB678VYW      0.846154     False
sub-NDARAG788YV9      0.846154     False
sub-NDARAK772VFJ      0.846154     False
sub-NDARAM946HJE      0.846154     False
sub-NDARAP283ZBW      0.846154     False
sub-NDARAY761BFH      0.846154     False
sub-NDARAY977BZT      0.846154     False
sub-NDARAZ532KK0      0.846154     False
sub-NDARBB542URX      0.846154     False
sub-NDARBF402JLH      0.846154     False
sub-NDARBH275EXT      0.846154     False


In [50]:
# Count availability per task among selected subjects
task_counts = {}
for col in task_columns:
    task_counts[col] = (selected_subjects[col] != 'unavailable').sum()

task_summary = pd.DataFrame.from_dict(task_counts, orient='index', columns=['n_selected_subjects'])
task_summary['percent_of_selected'] = (task_summary['n_selected_subjects'] / len(selected_subjects)) * 100
task_summary = task_summary.sort_values('percent_of_selected', ascending=False)

print(f"\nTask availability among selected subjects (n={len(selected_subjects)}):")
print(task_summary)


Task availability among selected subjects (n=11):
                           n_selected_subjects  percent_of_selected
RestingState                                11                100.0
DespicableMe                                11                100.0
FunwithFractals                             11                100.0
ThePresent                                  11                100.0
DiaryOfAWimpyKid                            11                100.0
contrastChangeDetection_1                   11                100.0
contrastChangeDetection_2                   11                100.0
contrastChangeDetection_3                   11                100.0
surroundSupp_1                              11                100.0
surroundSupp_2                              11                100.0
symbolSearch                                11                100.0
seqLearning6target                           0                  0.0
seqLearning8target                           0                  0

In [54]:
filtered_df = df[df['participant_id'].isin(SUBJECTS)]
len(filtered_df)

30

In [55]:
import pandas as pd

# Filter subjects first
filtered_df = df[df['participant_id'].isin(SUBJECTS)]

# Define active and passive tasks
ACTIVE_TASKS = {"contrastChangeDetection", "symbolSearch", "surroundSupp", "DespicableMe",
                "DiaryOfAWimpyKid", "FunwithFractals", "ThePresent"}
PASSIVE_TASKS = {"RestingState"}

# Define which task columns have _1, _2, _3 variants
task_variants = {
    'contrastChangeDetection': ['contrastChangeDetection_1', 'contrastChangeDetection_2', 'contrastChangeDetection_3'],
    'surroundSupp': ['surroundSupp_1', 'surroundSupp_2']
}

# Single column tasks (no variants)
single_column_tasks = list(ACTIVE_TASKS - {'contrastChangeDetection', 'surroundSupp'}) + list(PASSIVE_TASKS)

# Create a new DataFrame for processed availability
df_availability = filtered_df.copy()

# Process OR groups: if at least one variant is NOT 'unavailable', count as available
for task_name, variants in task_variants.items():
    df_availability[task_name + '_available'] = (df_availability[variants] != 'unavailable').any(axis=1).astype(int)

# Process single column tasks
for task in single_column_tasks:
    df_availability[task + '_available'] = (df_availability[task] != 'unavailable').astype(int)

# Count availability per task among subjects
task_counts = {}

for task in single_column_tasks:
    col = task + '_available'
    task_counts[task] = df_availability[col].sum()

for task_name in task_variants.keys():
    col = task_name + '_available'
    task_counts[task_name] = df_availability[col].sum()

# Create summary DataFrame
task_summary = pd.DataFrame.from_dict(task_counts, orient='index', columns=['n_subjects'])
task_summary['percent'] = (task_summary['n_subjects'] / len(filtered_df)) * 100

# Output only percentages
for task, percentage in task_summary['percent'].items():
    print(f"{task}: {percentage:.1f}%")

FunwithFractals: 90.0%
DiaryOfAWimpyKid: 90.0%
ThePresent: 83.3%
DespicableMe: 90.0%
symbolSearch: 76.7%
RestingState: 70.0%
contrastChangeDetection: 76.7%
surroundSupp: 90.0%


In [56]:
import pandas as pd

# Filter subjects first
filtered_df = df[df['participant_id'].isin(SUBJECTS)]

# Define active and passive tasks
ACTIVE_TASKS = {"contrastChangeDetection", "symbolSearch", "surroundSupp", "DespicableMe",
                "DiaryOfAWimpyKid", "FunwithFractals", "ThePresent"}
PASSIVE_TASKS = {"RestingState"}

# Define which task columns have _1, _2, _3 variants
task_variants = {
    'contrastChangeDetection': ['contrastChangeDetection_1', 'contrastChangeDetection_2', 'contrastChangeDetection_3'],
    'surroundSupp': ['surroundSupp_1', 'surroundSupp_2']
}

# Single column tasks (no variants)
single_column_tasks = list(ACTIVE_TASKS - {'contrastChangeDetection', 'surroundSupp'}) + list(PASSIVE_TASKS)

# Create a new DataFrame for processed availability
df_availability = filtered_df.copy()

# Process OR groups: if at least one variant is NOT 'unavailable', count as available
for task_name, variants in task_variants.items():
    df_availability[task_name + '_available'] = (df_availability[variants] != 'unavailable').any(axis=1).astype(int)

# Process single column tasks
for task in single_column_tasks:
    df_availability[task + '_available'] = (df_availability[task] != 'unavailable').astype(int)

# List all availability columns
availability_columns = [task + '_available' for task in single_column_tasks] + [task + '_available' for task in task_variants.keys()]

# Calculate percentage per subject (how many tasks available out of total)
df_availability['percent_available'] = df_availability[availability_columns].mean(axis=1) * 100

# Output percentages per subject
print("Percentages per subject:")
for idx, row in df_availability.iterrows():
    print(f"{row['participant_id']}: {row['percent_available']:.1f}%")

Percentages per subject:
sub-NDARAB678VYW: 100.0%
sub-NDARAB683CYD: 87.5%
sub-NDARAC296UCB: 75.0%
sub-NDARAD459XJK: 12.5%
sub-NDARAG429CGW: 37.5%
sub-NDARAG788YV9: 100.0%
sub-NDARAJ182PTB: 50.0%
sub-NDARAJ401AX3: 87.5%
sub-NDARAJ674WJT: 100.0%
sub-NDARAK772VFJ: 100.0%
sub-NDARAM177DKJ: 50.0%
sub-NDARAM946HJE: 100.0%
sub-NDARAN302EEM: 62.5%
sub-NDARAP283ZBW: 100.0%
sub-NDARAP748AWX: 62.5%
sub-NDARAU517MC6: 87.5%
sub-NDARAW427GWK: 75.0%
sub-NDARAW620GJ8: 87.5%
sub-NDARAY761BFH: 100.0%
sub-NDARAY977BZT: 100.0%
sub-NDARAZ532KK0: 100.0%
sub-NDARBB542URX: 100.0%
sub-NDARBE096YK6: 100.0%
sub-NDARBF402JLH: 100.0%
sub-NDARBH275EXT: 100.0%
sub-NDARBH482JK1: 75.0%
sub-NDARBH789CUP: 75.0%
sub-NDARBK194FF5: 100.0%
sub-NDARBP713ZWA: 87.5%
sub-NDARBT780KVC: 87.5%


In [58]:
df_availability['percent_available'].median(), df_availability['percent_available'].mean()

(np.float64(87.5), np.float64(83.33333333333333))

In [61]:
df_availability[df_availability['percent_available'] == 100]['participant_id'].to_list()

['sub-NDARAB678VYW',
 'sub-NDARAG788YV9',
 'sub-NDARAJ674WJT',
 'sub-NDARAK772VFJ',
 'sub-NDARAM946HJE',
 'sub-NDARAP283ZBW',
 'sub-NDARAY761BFH',
 'sub-NDARAY977BZT',
 'sub-NDARAZ532KK0',
 'sub-NDARBB542URX',
 'sub-NDARBE096YK6',
 'sub-NDARBF402JLH',
 'sub-NDARBH275EXT',
 'sub-NDARBK194FF5']

# train on 100% availability

## subject selection

In [62]:
import os
import numpy as np
import mne

ROOT = "/Users/roman/PycharmProjects/brain_data/ds005516"
WINDOW_SEC = 2.0
STEP_SEC = 2.0
TEST_RATIO = 0.3
SEED = 42
RESAMPLE_RATE = 125 # was 500

SUBJECTS = [
    "sub-NDARAB678VYW","sub-NDARAB683CYD","sub-NDARAC296UCB","sub-NDARAD459XJK",
    "sub-NDARAG429CGW","sub-NDARAG788YV9","sub-NDARAJ182PTB","sub-NDARAJ401AX3",
    "sub-NDARAJ674WJT","sub-NDARAK772VFJ","sub-NDARAM177DKJ","sub-NDARAM946HJE",
    "sub-NDARAN302EEM","sub-NDARAP283ZBW","sub-NDARAP748AWX","sub-NDARAU517MC6",
    "sub-NDARAW427GWK","sub-NDARAW620GJ8","sub-NDARAY761BFH","sub-NDARAY977BZT",
    "sub-NDARAZ532KK0","sub-NDARBB542URX","sub-NDARBE096YK6","sub-NDARBF402JLH",
    "sub-NDARBH275EXT","sub-NDARBH482JK1","sub-NDARBH789CUP","sub-NDARBK194FF5",
    "sub-NDARBP713ZWA","sub-NDARBT780KVC"
]

ACTIVE_TASKS = {"contrastChangeDetection", "symbolSearch", "surroundSupp","DespicableMe",
    "DiaryOfAWimpyKid","FunwithFractals","ThePresent"}
PASSIVE_TASKS = {
    "RestingState",
}
ALL_TASKS = ACTIVE_TASKS | PASSIVE_TASKS


# ------------------------------------------------------------
# split subjects
# ------------------------------------------------------------
def split_subjects(subjects):
    # subjects = np.array(subjects)
    # rng = np.random.RandomState(SEED)
    # rng.shuffle(subjects)
    #
    # n_test = int(len(subjects) * TEST_RATIO)
    # return subjects[n_test:], subjects[:n_test]

    return (
        df_availability[df_availability['percent_available'] == 100]['participant_id'].to_list(),  # 14 total
        df_availability[df_availability['percent_available'] != 100]['participant_id'].to_list()  # 16 total
    )



# ------------------------------------------------------------
# file handling
# ------------------------------------------------------------
def list_eeg_files(subject):
    eeg_dir = os.path.join(ROOT, subject, "eeg")
    if not os.path.exists(eeg_dir):
        return []

    return [
        os.path.join(eeg_dir, f)
        for f in os.listdir(eeg_dir)
        if f.endswith("_eeg.set")
    ]


def parse_task(path):
    for part in os.path.basename(path).split("_"):
        if part.startswith("task-"):
            return part.replace("task-", "")
    return None


def select_run1(files):
    groups = {}

    for f in files:
        task = parse_task(f)
        if task:
            groups.setdefault(task, []).append(f)

    selected = []
    for task, g in groups.items():
        run1 = [x for x in g if "run-1" in x]
        selected.append(run1[0] if run1 else sorted(g)[0])

    return selected


def get_label(task):
    if task in ACTIVE_TASKS:
        return 1
    if task in PASSIVE_TASKS:
        return 0
    return None


# ------------------------------------------------------------
# window extraction (NO processing)
# ------------------------------------------------------------
def extract_windows(raw, label, max_windows=50):
    raw.load_data()
    raw.resample(RESAMPLE_RATE)

    sfreq = raw.info["sfreq"]
    win = int(WINDOW_SEC * sfreq)
    step = int(STEP_SEC * sfreq)

    data = raw.get_data()   # (channels, time)

    X, y = [], []

    count = 0

    for start in range(0, data.shape[1] - win, step):
        window = data[:, start:start + win]   # (channels, time)

        X.append(window)
        y.append(label)

        count += 1
        if count >= max_windows:
            break

    return X, y


# ------------------------------------------------------------
# dataset builder
# ------------------------------------------------------------
def build_dataset(subjects):
    X_all, y_all = [], []

    for sub in subjects:
        eeg_files = select_run1(list_eeg_files(sub))

        for path in eeg_files:
            task = parse_task(path)
            label = get_label(task)

            if task not in ALL_TASKS or label is None:
                continue

            try:
                raw = mne.io.read_raw_eeglab(path, preload=False)

                X, y = extract_windows(raw, label)

                X_all.extend(X)
                y_all.extend(y)

                del raw

            except Exception as e:
                print("Skip:", path, e)

    return np.array(X_all), np.array(y_all)


# ------------------------------------------------------------
# run
# ------------------------------------------------------------
train_subjects, test_subjects = split_subjects(SUBJECTS)

X_train, y_train = build_dataset(train_subjects)
X_test, y_test   = build_dataset(test_subjects)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-DespicableMe_eeg.set
Reading 0 ... 86139  =      0.000 ...   172.278 secs...


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_47999/2323119315.py:142: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=False)


Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-ThePresent_eeg.set
Reading 0 ... 102439  =      0.000 ...   204.878 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-FunwithFractals_eeg.set
Reading 0 ... 82352  =      0.000 ...   164.704 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-contrastChangeDetection_run-1_eeg.set


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_47999/2323119315.py:142: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=False)


Reading 0 ... 179228  =      0.000 ...   358.456 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-surroundSupp_run-1_eeg.set
Reading 0 ... 139316  =      0.000 ...   278.632 secs...


...

Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARBT780KVC/eeg/sub-NDARBT780KVC_task-symbolSearch_eeg.set
Reading 0 ... 87287  =      0.000 ...   174.574 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARBT780KVC/eeg/sub-NDARBT780KVC_task-contrastChangeDetection_run-1_eeg.set
Reading 0 ... 121849  =      0.000 ...   243.698 secs...


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_47999/2323119315.py:142: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=False)


Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARBT780KVC/eeg/sub-NDARBT780KVC_task-ThePresent_eeg.set
Reading 0 ... 102295  =      0.000 ...   204.590 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARBT780KVC/eeg/sub-NDARBT780KVC_task-FunwithFractals_eeg.set
Reading 0 ... 82325  =      0.000 ...   164.650 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARBT780KVC/eeg/sub-NDARBT780KVC_task-DespicableMe_eeg.set
Reading 0 ... 86041  =      0.000 ...   172.082 secs...


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_47999/2323119315.py:142: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=False)
/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_47999/2323119315.py:142: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=False)


Train: (5347, 129, 250) (5347,)
Test : (4118, 129, 250) (4118,)


In [64]:
import numpy as np

eps = 1e-8
flat_idx_common = set()

for i in range(len(X_test)):
    stds = np.std(X_test[i], axis=1)
    flat_idx = set(np.where(stds < eps)[0])

    flat_idx_common |= flat_idx

flat_idx_common_test = set()

for i in range(len(X_test)):
    stds = np.std(X_test[i], axis=1)
    flat_idx = set(np.where(stds < eps)[0])

    flat_idx_common_test |= flat_idx

bad_channels = np.array(sorted(flat_idx_common_test | flat_idx_common))

# build boolean mask: True = keep
n_channels = X_train.shape[1]
keep_mask = np.ones(n_channels, dtype=bool)
keep_mask[bad_channels] = False

# apply to both splits
X_train_fixed = X_train[:, keep_mask, :]
X_test_fixed  = X_test[:,  keep_mask, :]

print("Original shape:", X_train.shape)
print("Fixed shape   :", X_train_fixed.shape)

print("Original shape:", X_test.shape)
print("Fixed shape   :", X_test_fixed.shape)

Original shape: (5347, 129, 250)
Fixed shape   : (5347, 117, 250)
Original shape: (4118, 129, 250)
Fixed shape   : (4118, 117, 250)


## pipeline - chosen best, cov

In [65]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

from pyriemann.utils.mean import mean_covariance
from pyriemann.tangentspace import TangentSpace
from scipy.linalg import fractional_matrix_power

REG = 1e-5

# ============================================================
# 1. COVARIANCE FROM WINDOW SIGNALS (IMPORTANT PART)
# ============================================================

def compute_covariance(w):
    # w: (channels, time)
    w = w - np.mean(w, axis=1, keepdims=True)

    s = w.shape[1]
    C = (w @ w.T) / (s - 1)

    C += REG * np.eye(C.shape[0])
    return C


# ============================================================
# 2. BUILD COVARIANCES FROM FIXED WINDOWS
# ============================================================

def build_covariances(X):
    covs = []

    for w in X:
        # w: (channels, time)
        if np.any(np.std(w, axis=1) < 1e-10):
            continue

        C = compute_covariance(w)

        if np.any(~np.isfinite(C)):
            continue

        covs.append(C)

    return np.array(covs)


# ============================================================
# 3. BUILD DATASETS
# ============================================================

covs_train = build_covariances(X_train_fixed)
covs_test  = build_covariances(X_test_fixed)

y_train = y_train[:len(covs_train)]
y_test  = y_test[:len(covs_test)]


# ============================================================
# 4. RIEMANNIAN MEAN + WHITENING
# ============================================================

M = mean_covariance(covs_train, metric='riemann')

def whiten(C, M):
    M_inv_sqrt = fractional_matrix_power(M, -0.5)
    return M_inv_sqrt @ C @ M_inv_sqrt


covs_train_w = np.array([whiten(C, M) for C in covs_train])
covs_test_w  = np.array([whiten(C, M) for C in covs_test])


# ============================================================
# 5. TANGENT SPACE EMBEDDING
# ============================================================

ts = TangentSpace(metric='riemann')
ts.fit(covs_train_w)

X_ts_train = ts.transform(covs_train_w)
X_ts_test  = ts.transform(covs_test_w)


# ============================================================
# 6. MODEL
# ============================================================

model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42
)

model.fit(X_ts_train, y_train)

y_pred = model.predict(X_ts_test)
y_prob = model.predict_proba(X_ts_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       350
           1       0.92      1.00      0.96      3768

    accuracy                           0.92      4118
   macro avg       0.46      0.50      0.48      4118
weighted avg       0.84      0.92      0.87      4118

ROC-AUC: 0.6126895662723688


/Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

# paper-like on fixed

In [7]:
import numpy as np
from scipy.linalg import fractional_matrix_power
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

# ============================================================
# PS-ITSA CONFIG
# ============================================================
REG = 1e-5
CALIBRATION_RATIO = 0.3
VARIANCE_THRESHOLD = 0.999

# ============================================================
# COVARIANCE ESTIMATION
# ============================================================
def compute_covariance(w):
    """Compute regularized covariance matrix from window"""
    w = w - np.mean(w, axis=1, keepdims=True)
    s = w.shape[1]
    C = (w @ w.T) / (s - 1)
    C += REG * np.eye(C.shape[0])
    return C

# ============================================================
# HJORTH FEATURES (TIME-DOMAIN)
# ============================================================
def hjorth_activity(x):
    return np.var(x)

def hjorth_mobility(x):
    v = np.var(x)
    return np.sqrt(np.var(np.diff(x)) / (v + 1e-12))

def hjorth_complexity(x):
    m = hjorth_mobility(x)
    return hjorth_mobility(np.diff(x)) / (m + 1e-12)

def extract_hjorth_features(data, ch_names=None):
    """
    Extract Hjorth parameters for all channels.
    data: (channels, time)
    Returns: 1D array of features
    """
    n_channels = data.shape[0]
    feats = []

    for i in range(n_channels):
        x = data[i]

        # Original Hjorth parameters
        a = hjorth_activity(x)
        m = hjorth_mobility(x)
        c = hjorth_complexity(x)

        feats.extend([a, m, c, np.log(a + 1e-12), np.log(m + 1e-12)])

    return np.array(feats)

# ============================================================
# LOG-EUCLIDEAN GEOMETRY
# ============================================================
def log_euclidean(C):
    """Matrix logarithm for SPD matrix"""
    eigvals, eigvecs = np.linalg.eigh(C)
    eigvals = np.clip(eigvals, 1e-12, None)
    return eigvecs @ np.diag(np.log(eigvals)) @ eigvecs.T

def log_euclidean_mean(covs):
    """Log-Euclidean mean of SPD matrices"""
    logs = np.array([log_euclidean(C) for C in covs])
    mean_log = np.mean(logs, axis=0)
    eigvals, eigvecs = np.linalg.eigh(mean_log)
    eigvals = np.clip(eigvals, -1e12, None)
    return eigvecs @ np.diag(np.exp(eigvals)) @ eigvecs.T

def tangent_space_projection(covs, reference):
    """Project SPD matrices to tangent space at reference"""
    ref_inv_sqrt = fractional_matrix_power(reference, -0.5)

    projected = []
    for C in covs:
        C_rec = ref_inv_sqrt @ C @ ref_inv_sqrt
        C_ts = log_euclidean(C_rec)
        C_ts = (C_ts + C_ts.T) / 2  # ensure symmetry
        projected.append(C_ts)

    return np.array(projected)

# ============================================================
# RESCALING
# ============================================================
def rescale_to_unit_mean_norm(features):
    """Normalize average Frobenius norm to 1"""
    norms = np.sqrt(np.sum(features.reshape(len(features), -1)**2, axis=1))
    mean_norm = np.mean(norms)
    return features / (mean_norm + 1e-12)

# ============================================================
# CLASS-WISE ANCHOR POINTS
# ============================================================
def class_wise_anchors_matrix(features, labels):
    """Return anchors as matrices (ch, ch*K)"""
    unique_classes = np.unique(labels)
    anchors = []

    for c in unique_classes:
        class_features = features[labels == c]
        anchor = np.mean(class_features, axis=0)
        anchors.append(anchor)

    return np.hstack(anchors)

def compute_rotation_matrix_channel_space(anchors_train, anchors_calib, variance_threshold=0.999):
    """Compute rotation matrix R = Ũ @ Ṽ^T"""
    C_TC = anchors_train @ anchors_calib.T
    U, S, Vt = np.linalg.svd(C_TC, full_matrices=False)

    var_exp = np.cumsum(S) / (np.sum(S) + 1e-12)
    n_components = np.searchsorted(var_exp, variance_threshold) + 1

    U_trunc = U[:, :n_components]
    Vt_trunc = Vt[:n_components, :]

    return U_trunc @ Vt_trunc

def apply_rotation_to_matrices(matrices, rotation_matrix):
    """Apply rotation to each matrix: R @ X"""
    return np.array([rotation_matrix @ X for X in matrices])

# ============================================================
# MAIN PS-ITSA + HJORTH PIPELINE
# ============================================================
def ps_itsa_with_hjorth(covs_train, covs_test,
                         hjorth_train, hjorth_test,
                         y_train, y_test,
                         calib_ratio=CALIBRATION_RATIO,
                         var_threshold=VARIANCE_THRESHOLD):
    """
    Complete PS-ITSA implementation with Hjorth time-domain features.

    Inputs:
        covs_train: list/array of training covariance matrices (n_train, ch, ch)
        covs_test: list/array of test covariance matrices (n_test, ch, ch)
        hjorth_train: array of Hjorth features (n_train, n_hjorth_feats)
        hjorth_test: array of Hjorth features (n_test, n_hjorth_feats)
        y_train: training labels
        y_test: test labels
    """

    n_channels = covs_train[0].shape[0]
    print(f"\n{'='*60}")
    print(f"PS-ITSA + HJORTH PIPELINE")
    print(f"{'='*60}")
    print(f"Number of channels: {n_channels}")
    print(f"Training samples: {len(covs_train)}")
    print(f"Testing samples: {len(covs_test)}")

    # STEP 1: Recentering (Log-Euclidean)
    print("\n[Step 1] Recentering (Log-Euclidean)...")
    M = log_euclidean_mean(covs_train)
    C_train_ts = tangent_space_projection(covs_train, M)
    C_test_ts = tangent_space_projection(covs_test, M)
    print(f"  Training tangent space: {C_train_ts.shape}")
    print(f"  Testing tangent space: {C_test_ts.shape}")

    # STEP 2: Rescale
    print("\n[Step 2] Rescale (unit mean norm)...")
    C_train_rescaled = rescale_to_unit_mean_norm(C_train_ts)
    C_test_rescaled = rescale_to_unit_mean_norm(C_test_ts)

    # STEP 3: Rotation alignment
    print("\n[Step 3] Rotation alignment...")
    n_test = len(C_test_rescaled)
    n_calib = int(n_test * calib_ratio)

    np.random.seed(42)
    indices = np.random.permutation(n_test)
    calib_idx = indices[:n_calib]
    eval_idx = indices[n_calib:]

    C_calib = C_test_rescaled[calib_idx]
    C_eval = C_test_rescaled[eval_idx]
    y_calib = y_test[calib_idx]
    y_eval = y_test[eval_idx]

    hjorth_calib = hjorth_test[calib_idx]
    hjorth_eval = hjorth_test[eval_idx]

    print(f"  Calibration samples: {len(C_calib)}")
    print(f"  Evaluation samples: {len(C_eval)}")

    # Compute anchors
    anchors_train = class_wise_anchors_matrix(C_train_rescaled, y_train)
    anchors_calib = class_wise_anchors_matrix(C_calib, y_calib)

    # Compute rotation matrix
    R = compute_rotation_matrix_channel_space(anchors_train, anchors_calib, var_threshold)
    print(f"  Rotation matrix shape: {R.shape}")

    # Apply rotation
    C_eval_rotated = apply_rotation_to_matrices(C_eval, R)
    C_train_rotated = apply_rotation_to_matrices(C_train_rescaled, R)

    # Flatten covariance features
    X_cov_train = C_train_rotated.reshape(len(C_train_rotated), -1)
    X_cov_eval = C_eval_rotated.reshape(len(C_eval_rotated), -1)
    print(f"  Flattened covariance features: {X_cov_train.shape[1]}")

    # STEP 4: Fuse with Hjorth features
    print("\n[Step 4] Fusing Hjorth features...")
    scaler_hjorth = StandardScaler()
    hjorth_train_scaled = scaler_hjorth.fit_transform(hjorth_train)
    hjorth_calib_scaled = scaler_hjorth.transform(hjorth_calib)
    hjorth_eval_scaled = scaler_hjorth.transform(hjorth_eval)

    # Use ALL training data (not just calibration)
    X_train = np.concatenate([X_cov_train, hjorth_train_scaled], axis=1)
    X_test = np.concatenate([X_cov_eval, hjorth_eval_scaled], axis=1)

    print(f"  Hjorth features: {hjorth_train_scaled.shape[1]}")
    print(f"  Total features: {X_train.shape[1]}")

    return X_train, X_test, y_train, y_eval

# ============================================================
# MODEL EVALUATION
# ============================================================
def evaluate_with_xgboost(X_train, X_test, y_train, y_test, n_hjorth_features=None):
    """Train XGBoost and evaluate"""

    X_train = np.real(X_train).astype(np.float64)
    X_test = np.real(X_test).astype(np.float64)

    model = XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        n_jobs=-1,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print("\n" + "="*60)
    print("CLASSIFICATION RESULTS")
    print("="*60)
    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

    # Feature importance breakdown
    if n_hjorth_features is not None:
        print("\n" + "-"*40)
        print("FEATURE IMPORTANCE ANALYSIS")
        print("-"*40)

        importances = model.feature_importances_
        n_cov_features = X_train.shape[1] - n_hjorth_features
        cov_importance = np.mean(importances[:n_cov_features])
        hjorth_importance = np.mean(importances[n_cov_features:])

        print(f"\nFeature type summary:")
        print(f"  Covariance features ({n_cov_features}): {cov_importance:.4f}")
        print(f"  Hjorth features ({n_hjorth_features}): {hjorth_importance:.4f}")
        print(f"  Ratio (Hjorth/Cov): {hjorth_importance/cov_importance:.2f}x")

    return model, y_pred, y_prob

## usage

In [14]:

# Step 1: Compute covariance matrices
covs_train = np.array([compute_covariance(window) for window in X_train_fixed])
covs_test = np.array([compute_covariance(window) for window in X_test_fixed])

# Step 2: Extract Hjorth features
hjorth_train = np.array([extract_hjorth_features(window) for window in X_train_fixed])
hjorth_test = np.array([extract_hjorth_features(window) for window in X_test_fixed])

# Step 3: Run PS-ITSA pipeline
X_train, X_test, y_train_aligned, y_test_aligned = ps_itsa_with_hjorth(
    covs_train, covs_test,
    hjorth_train, hjorth_test,
    y_train, y_test
)

# Step 4: Evaluate
model, y_pred, y_prob = evaluate_with_xgboost(
    X_train, X_test,
    y_train_aligned, y_test_aligned,
    n_hjorth_features=hjorth_train.shape[1]
)


PS-ITSA + HJORTH PIPELINE
Number of channels: 103
Training samples: 6218
Testing samples: 3247

[Step 1] Recentering (Log-Euclidean)...
  Training tangent space: (6218, 103, 103)
  Testing tangent space: (3247, 103, 103)

[Step 2] Rescale (unit mean norm)...

[Step 3] Rotation alignment...
  Calibration samples: 974
  Evaluation samples: 2273
  Rotation matrix shape: (103, 103)
  Flattened covariance features: 10609

[Step 4] Fusing Hjorth features...
  Hjorth features: 515
  Total features: 11124

CLASSIFICATION RESULTS
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       247
           1       0.89      1.00      0.94      2026

    accuracy                           0.89      2273
   macro avg       0.45      0.50      0.47      2273
weighted avg       0.79      0.89      0.84      2273

ROC-AUC: 0.5543

----------------------------------------
FEATURE IMPORTANCE ANALYSIS
----------------------------------------

Feature type summ

## paper-like combined results

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       247
           1       0.89      1.00      0.94      2026

    accuracy                           0.89      2273
   macro avg       0.45      0.50      0.47      2273
weighted avg       0.79      0.89      0.84      2273

ROC-AUC: 0.5543

## only hjorth results

              precision    recall  f1-score   support

           0       0.04      0.01      0.01       347
           1       0.89      0.98      0.93      2900

    accuracy                           0.88      3247
   macro avg       0.46      0.49      0.47      3247
weighted avg       0.80      0.88      0.84      3247

ROC-AUC: 0.6434452946437443

## combined results with optimized covariance

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       347
           1       0.89      0.98      0.93      2900

    accuracy                           0.87      3247
   macro avg       0.45      0.49      0.47      3247
weighted avg       0.80      0.87      0.83      3247

ROC-AUC: 0.5976239689953294

# remove some tasks from active state

In [21]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

from pyriemann.utils.mean import mean_covariance
from pyriemann.tangentspace import TangentSpace
from scipy.linalg import fractional_matrix_power

REG = 1e-5

# ============================================================
# 1. HJORTH FEATURES (per window)
# ============================================================

def hjorth_activity(x):
    return np.var(x)

def hjorth_mobility(x):
    v = np.var(x)
    return np.sqrt(np.var(np.diff(x)) / (v + 1e-12))

def hjorth_complexity(x):
    m = hjorth_mobility(x)
    return hjorth_mobility(np.diff(x)) / (m + 1e-12)


def extract_hjorth(X):
    """
    X: (n_samples, n_channels, n_times)
    returns: (n_samples, n_channels * 5 features)
    """
    feats_all = []

    for w in X:
        feats = []

        for ch in range(w.shape[0]):
            x = w[ch]

            a = hjorth_activity(x)
            m = hjorth_mobility(x)
            c = hjorth_complexity(x)

            feats.extend([
                a,
                m,
                c,
                np.log(a + 1e-12),
                np.log(m + 1e-12)
            ])

        feats_all.append(feats)

    return np.array(feats_all)


# ============================================================
# 2. COVARIANCE FEATURES
# ============================================================

def compute_covariance(w):
    w = w - np.mean(w, axis=1, keepdims=True)
    s = w.shape[1]

    C = (w @ w.T) / (s - 1)
    C += REG * np.eye(C.shape[0])

    return C


def build_covs(X):
    covs = []

    for w in X:
        if np.any(np.std(w, axis=1) < 1e-10):
            continue
        covs.append(compute_covariance(w))

    return np.array(covs)


def whiten(C, M):
    M_inv_sqrt = fractional_matrix_power(M, -0.5)
    return M_inv_sqrt @ C @ M_inv_sqrt

In [15]:
import os
import numpy as np
import mne

ROOT = "/Users/roman/PycharmProjects/brain_data/ds005516"
WINDOW_SEC = 2.0
STEP_SEC = 2.0
TEST_RATIO = 0.3
SEED = 42
RESAMPLE_RATE = 125 # was 500

SUBJECTS = [
    "sub-NDARAB678VYW","sub-NDARAB683CYD","sub-NDARAC296UCB","sub-NDARAD459XJK",
    "sub-NDARAG429CGW","sub-NDARAG788YV9","sub-NDARAJ182PTB","sub-NDARAJ401AX3",
    "sub-NDARAJ674WJT","sub-NDARAK772VFJ","sub-NDARAM177DKJ","sub-NDARAM946HJE",
    "sub-NDARAN302EEM","sub-NDARAP283ZBW","sub-NDARAP748AWX","sub-NDARAU517MC6",
    "sub-NDARAW427GWK","sub-NDARAW620GJ8","sub-NDARAY761BFH","sub-NDARAY977BZT",
    "sub-NDARAZ532KK0","sub-NDARBB542URX","sub-NDARBE096YK6","sub-NDARBF402JLH",
    "sub-NDARBH275EXT","sub-NDARBH482JK1","sub-NDARBH789CUP","sub-NDARBK194FF5",
    "sub-NDARBP713ZWA","sub-NDARBT780KVC"
]

# ACTIVE_TASKS = {"contrastChangeDetection", "symbolSearch", "surroundSupp","DespicableMe",
#     "DiaryOfAWimpyKid","FunwithFractals","ThePresent"}
ACTIVE_TASKS = {"contrastChangeDetection", "symbolSearch"}
PASSIVE_TASKS = {
    "RestingState"
}
ALL_TASKS = ACTIVE_TASKS | PASSIVE_TASKS


# ------------------------------------------------------------
# split subjects
# ------------------------------------------------------------
def split_subjects(subjects):
    subjects = np.array(subjects)
    rng = np.random.RandomState(SEED)
    rng.shuffle(subjects)

    n_test = int(len(subjects) * TEST_RATIO)
    return subjects[n_test:], subjects[:n_test]


# ------------------------------------------------------------
# file handling
# ------------------------------------------------------------
def list_eeg_files(subject):
    eeg_dir = os.path.join(ROOT, subject, "eeg")
    if not os.path.exists(eeg_dir):
        return []

    return [
        os.path.join(eeg_dir, f)
        for f in os.listdir(eeg_dir)
        if f.endswith("_eeg.set")
    ]


def parse_task(path):
    for part in os.path.basename(path).split("_"):
        if part.startswith("task-"):
            return part.replace("task-", "")
    return None


def select_run1(files):
    groups = {}

    for f in files:
        task = parse_task(f)
        if task:
            groups.setdefault(task, []).append(f)

    selected = []
    for task, g in groups.items():
        run1 = [x for x in g if "run-1" in x]
        selected.append(run1[0] if run1 else sorted(g)[0])

    return selected


def get_label(task):
    if task in ACTIVE_TASKS:
        return 1
    if task in PASSIVE_TASKS:
        return 0
    return None


# ------------------------------------------------------------
# window extraction (NO processing)
# ------------------------------------------------------------
def extract_windows(raw, label, max_windows=50):
    raw.load_data()
    raw.resample(RESAMPLE_RATE)

    sfreq = raw.info["sfreq"]
    win = int(WINDOW_SEC * sfreq)
    step = int(STEP_SEC * sfreq)

    data = raw.get_data()   # (channels, time)

    X, y = [], []

    count = 0

    for start in range(0, data.shape[1] - win, step):
        window = data[:, start:start + win]   # (channels, time)

        X.append(window)
        y.append(label)

        count += 1
        if count >= max_windows:
            break

    return X, y


# ------------------------------------------------------------
# dataset builder
# ------------------------------------------------------------
def build_dataset(subjects):
    X_all, y_all = [], []

    for sub in subjects:
        eeg_files = select_run1(list_eeg_files(sub))

        for path in eeg_files:
            task = parse_task(path)
            label = get_label(task)

            if task not in ALL_TASKS or label is None:
                continue

            try:
                raw = mne.io.read_raw_eeglab(path, preload=False)

                X, y = extract_windows(raw, label)

                X_all.extend(X)
                y_all.extend(y)

                del raw

            except Exception as e:
                print("Skip:", path, e)

    return np.array(X_all), np.array(y_all)


# ------------------------------------------------------------
# run
# ------------------------------------------------------------
train_subjects, test_subjects = split_subjects(SUBJECTS)

X_train, y_train = build_dataset(train_subjects)
X_test, y_test   = build_dataset(test_subjects)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-contrastChangeDetection_run-1_eeg.set
Reading 0 ... 179228  =      0.000 ...   358.456 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-symbolSearch_eeg.set
Reading 0 ... 82994  =      0.000 ...   165.988 secs...


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_31693/1813185075.py:137: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=False)


Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAB678VYW/eeg/sub-NDARAB678VYW_task-RestingState_eeg.set
Reading 0 ... 172890  =      0.000 ...   345.780 secs...


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_31693/1813185075.py:137: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=False)


Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARAG429CGW/eeg/sub-NDARAG429CGW_task-symbolSearch_eeg.set
Reading 0 ... 85818  =      0.000 ...   171.636 secs...


...

Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARBP713ZWA/eeg/sub-NDARBP713ZWA_task-contrastChangeDetection_run-1_eeg.set
Reading 0 ... 155213  =      0.000 ...   310.426 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARBP713ZWA/eeg/sub-NDARBP713ZWA_task-symbolSearch_eeg.set
Reading 0 ... 113163  =      0.000 ...   226.326 secs...


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_31693/1813185075.py:137: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=False)


Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARBH275EXT/eeg/sub-NDARBH275EXT_task-contrastChangeDetection_run-1_eeg.set
Reading 0 ... 239671  =      0.000 ...   479.342 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARBH275EXT/eeg/sub-NDARBH275EXT_task-RestingState_eeg.set
Reading 0 ... 69057  =      0.000 ...   138.114 secs...
Reading /Users/roman/PycharmProjects/brain_data/ds005516/sub-NDARBH275EXT/eeg/sub-NDARBH275EXT_task-symbolSearch_eeg.set
Reading 0 ... 91241  =      0.000 ...   182.482 secs...


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_31693/1813185075.py:137: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=False)


Train: (1900, 129, 250) (1900,)
Test : (1047, 129, 250) (1047,)


In [16]:
import numpy as np

eps = 1e-8
flat_idx_common = set()

for i in range(len(X_train)):
    stds = np.std(X_train[i], axis=1)
    flat_idx = set(np.where(stds < eps)[0])

    flat_idx_common |= flat_idx

print(sorted(flat_idx_common))

[np.int64(0), np.int64(7), np.int64(9), np.int64(10), np.int64(20), np.int64(24), np.int64(28), np.int64(31), np.int64(55), np.int64(89), np.int64(105), np.int64(106), np.int64(121), np.int64(126), np.int64(128)]


In [17]:
import numpy as np

eps = 1e-8
flat_idx_common_test = set()

for i in range(len(X_test)):
    stds = np.std(X_test[i], axis=1)
    flat_idx = set(np.where(stds < eps)[0])

    flat_idx_common_test |= flat_idx

print(sorted(flat_idx_common_test))

[np.int64(2), np.int64(3), np.int64(9), np.int64(16), np.int64(42), np.int64(44), np.int64(47), np.int64(48), np.int64(55), np.int64(79), np.int64(89), np.int64(106), np.int64(112), np.int64(118), np.int64(121), np.int64(125), np.int64(128)]


In [18]:
bad_channels = np.array(sorted(flat_idx_common_test | flat_idx_common))

# build boolean mask: True = keep
n_channels = X_train.shape[1]
keep_mask = np.ones(n_channels, dtype=bool)
keep_mask[bad_channels] = False

# apply to both splits
X_train_fixed = X_train[:, keep_mask, :]
X_test_fixed  = X_test[:,  keep_mask, :]

print("Original shape:", X_train.shape)
print("Fixed shape   :", X_train_fixed.shape)

Original shape: (1900, 129, 250)
Fixed shape   : (1900, 103, 250)


## paper

In [19]:
# Step 1: Compute covariance matrices
covs_train = np.array([compute_covariance(window) for window in X_train_fixed])
covs_test = np.array([compute_covariance(window) for window in X_test_fixed])

# Step 2: Extract Hjorth features
hjorth_train = np.array([extract_hjorth_features(window) for window in X_train_fixed])
hjorth_test = np.array([extract_hjorth_features(window) for window in X_test_fixed])

# Step 3: Run PS-ITSA pipeline
X_train, X_test, y_train_aligned, y_test_aligned = ps_itsa_with_hjorth(
    covs_train, covs_test,
    hjorth_train, hjorth_test,
    y_train, y_test
)

# Step 4: Evaluate
model, y_pred, y_prob = evaluate_with_xgboost(
    X_train, X_test,
    y_train_aligned, y_test_aligned,
    n_hjorth_features=hjorth_train.shape[1]
)


PS-ITSA + HJORTH PIPELINE
Number of channels: 103
Training samples: 1900
Testing samples: 1047

[Step 1] Recentering (Log-Euclidean)...
  Training tangent space: (1900, 103, 103)
  Testing tangent space: (1047, 103, 103)

[Step 2] Rescale (unit mean norm)...

[Step 3] Rotation alignment...
  Calibration samples: 314
  Evaluation samples: 733
  Rotation matrix shape: (103, 103)
  Flattened covariance features: 10609

[Step 4] Fusing Hjorth features...
  Hjorth features: 515
  Total features: 11124

CLASSIFICATION RESULTS
              precision    recall  f1-score   support

           0       0.39      0.67      0.49       256
           1       0.71      0.43      0.53       477

    accuracy                           0.51       733
   macro avg       0.55      0.55      0.51       733
weighted avg       0.60      0.51      0.52       733

ROC-AUC: 0.5150

----------------------------------------
FEATURE IMPORTANCE ANALYSIS
----------------------------------------

Feature type summa

## optimized

In [22]:
# ============================================================
# 3. HJORTH FEATURES
# ============================================================

X_h_train = extract_hjorth(X_train_fixed)
X_h_test  = extract_hjorth(X_test_fixed)

scaler = StandardScaler()
X_h_train = scaler.fit_transform(X_h_train)
X_h_test  = scaler.transform(X_h_test)


# ============================================================
# 4. RIEMANNIAN FEATURES
# ============================================================

covs_train = build_covs(X_train_fixed)
covs_test  = build_covs(X_test_fixed)

# IMPORTANT: ensure alignment (same number of samples)
n = min(len(covs_train), len(covs_test))
covs_train = covs_train[:n]
covs_test  = covs_test[:n]

y_train = y_train[:n]
y_test  = y_test[:n]

# Riemann mean
M = mean_covariance(covs_train, metric='riemann')

covs_train_w = np.array([whiten(C, M) for C in covs_train])
covs_test_w  = np.array([whiten(C, M) for C in covs_test])

ts = TangentSpace(metric='riemann')
ts.fit(covs_train_w)

X_ts_train = ts.transform(covs_train_w)
X_ts_test  = ts.transform(covs_test_w)


# ============================================================
# 5. FUSION (CRITICAL STEP)
# ============================================================

X_train = np.concatenate([X_h_train[:n], X_ts_train], axis=1)
X_test  = np.concatenate([X_h_test[:n],  X_ts_test], axis=1)


# ============================================================
# 6. MODEL
# ============================================================

model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.67      0.10      0.18       347
           1       0.69      0.97      0.81       700

    accuracy                           0.69      1047
   macro avg       0.68      0.54      0.49      1047
weighted avg       0.68      0.69      0.60      1047

ROC-AUC: 0.6055990119390696


## regular features

In [20]:
import numpy as np
import mne

from scipy.signal import welch
from scipy.stats import entropy
from itertools import combinations

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.covariance import LedoitWolf

from xgboost import XGBClassifier

from pyriemann.utils.mean import mean_covariance
from pyriemann.tangentspace import TangentSpace
from scipy.linalg import fractional_matrix_power

def compute_covariance(w):
    w = w - np.mean(w, axis=1, keepdims=True)

    s = w.shape[1]
    C = (w @ w.T) / (s - 1)

    # small ridge regularization (NOT full Ledoit collapse)
    C += REG * np.eye(C.shape[0])

    return C

def hjorth_activity(x): return np.var(x)

def hjorth_mobility(x):
    v = np.var(x)
    return np.sqrt(np.var(np.diff(x)) / (v + 1e-12))

def hjorth_complexity(x):
    m = hjorth_mobility(x)
    return hjorth_mobility(np.diff(x)) / (m + 1e-12)

def extract_features_window(data, ch_names):
    feats = {}

    for i, ch in enumerate(ch_names):
        x = data[i]

        feats[f"{ch}_a"] = hjorth_activity(x)
        feats[f"{ch}_m"] = hjorth_mobility(x)
        feats[f"{ch}_c"] = hjorth_complexity(x)

        feats[f"{ch}_log_a"] = np.log(feats[f"{ch}_a"] + 1e-12)
        feats[f"{ch}_log_m"] = np.log(feats[f"{ch}_m"] + 1e-12)

    return feats

def whiten(C, M):
    M_inv_sqrt = fractional_matrix_power(M, -0.5)
    return M_inv_sqrt @ C @ M_inv_sqrt

# ============================================================
# FEATURE EXTRACTION FROM PRECOMPUTED WINDOWS
# ============================================================

def extract_feature_matrix(X):
    """
    X: (n_samples, n_channels, n_times)
    returns: (n_samples, n_features)
    """
    all_feats = []

    for w in X:

        feats = extract_features_window(w, [f"ch{i}" for i in range(w.shape[0])])
        vals = np.array(list(feats.values()))

        if not np.all(np.isfinite(vals)):
            continue

        all_feats.append(vals)

    return np.array(all_feats)


# compute features
X_train_feats = extract_feature_matrix(X_train_fixed)
X_test_feats  = extract_feature_matrix(X_test_fixed)

print("Feature shapes:")
print("Train:", X_train_feats.shape)
print("Test :", X_test_feats.shape)

model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42
)

model.fit(X_train_feats, y_train)

y_pred = model.predict(X_test_feats)
y_prob = model.predict_proba(X_test_feats)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Feature shapes:
Train: (1900, 515)
Test : (1047, 515)
              precision    recall  f1-score   support

           0       0.34      0.31      0.33       347
           1       0.67      0.69      0.68       700

    accuracy                           0.57      1047
   macro avg       0.50      0.50      0.50      1047
weighted avg       0.56      0.57      0.56      1047

ROC-AUC: 0.5373198847262248


## all results

### regular feats

              precision    recall  f1-score   support

           0       0.34      0.31      0.33       347
           1       0.67      0.69      0.68       700

    accuracy                           0.57      1047
   macro avg       0.50      0.50      0.50      1047
weighted avg       0.56      0.57      0.56      1047

ROC-AUC: 0.5373198847262248

### optimized

              precision    recall  f1-score   support

           0       0.67      0.10      0.18       347
           1       0.69      0.97      0.81       700

    accuracy                           0.69      1047
   macro avg       0.68      0.54      0.49      1047
weighted avg       0.68      0.69      0.60      1047

ROC-AUC: 0.6055990119390696

### paper

              precision    recall  f1-score   support

           0       0.39      0.67      0.49       256
           1       0.71      0.43      0.53       477

    accuracy                           0.51       733
   macro avg       0.55      0.55      0.51       733
weighted avg       0.60      0.51      0.52       733

ROC-AUC: 0.5150